In [1]:
print("hello world")

hello world


In [2]:
pip install pylint

   ---------------------------------------- 0.0/536.4 kB ? eta -:--:--
   -- ------------------------------------ 30.7/536.4 kB 640.0 kB/s eta 0:00:01
   ------ --------------------------------- 92.2/536.4 kB 1.0 MB/s eta 0:00:01
   ------------ --------------------------- 163.8/536.4 kB 1.2 MB/s eta 0:00:01
   ------------------- -------------------- 256.0/536.4 kB 1.6 MB/s eta 0:00:01
   ----------------------- ---------------- 317.4/536.4 kB 1.4 MB/s eta 0:00:01
   --------------------------- ------------ 368.6/536.4 kB 1.5 MB/s eta 0:00:01
   ----------------------------- ---------- 389.1/536.4 kB 1.3 MB/s eta 0:00:01
   ------------------------------------- -- 501.8/536.4 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 536.4/536.4 kB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/276.4 kB ? eta -:--:--
   ---------- ----------------------------- 71.7/276.4 kB 2.0 MB/s eta 0:00:01
   ---------------------- ----------------- 153.6/276.4 kB 1

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python311\\Scripts\\isort.exe' -> 'c:\\Python311\\Scripts\\isort.exe.deleteme'


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import cv2
import numpy as np
import re
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

class ScannedPDFToExcel:
    def __init__(self, tesseract_path=None):
        """
        Initialize the OCR processor
        """
        # Set tesseract path if provided (Windows users might need this)
        if tesseract_path:
            pytesseract.pytesseract.tesseract_cmd = tesseract_path
        
        # Define expected columns
        self.expected_columns = [
            'DATE', 'SOURCE', 'DESCRIPTION', 'REFERENCE', 'CURRENCY',
            'DEBIT(SOURCE)', 'CREDIT(SOURCE)', 'DEBIT(USD)', 'CREDIT(USD)',
            'RUNNING BALANCE(USD)', 'RELATED ACCOUNT'
        ]
        
        # Account patterns to identify
        self.account_patterns = [
            r'Compensation.*?Zim.*?Account',
            r'Insurance.*?Proceeds.*?Zim.*?Account',
            r'Zim.*?Account',
            r'[A-Z][a-z]+.*?Account'
        ]

    def preprocess_image(self, image):
        """
        Preprocess image for better OCR accuracy
        """
        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        # Apply thresholding to get binary image
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
        
        # Denoise
        denoised = cv2.fastNlMeansDenoising(thresh)
        
        # Apply dilation to connect text components
        kernel = np.ones((1, 1), np.uint8)
        dilated = cv2.dilate(denoised, kernel, iterations=1)
        
        return dilated

    def extract_text_from_scanned_pdf(self, pdf_path, dpi=300):
        """
        Extract text from scanned PDF using OCR
        """
        print(f"Converting PDF to images with {dpi} DPI...")
        
        # Convert PDF to images
        images = convert_from_path(pdf_path, dpi=dpi)
        
        all_text = []
        page_accounts = []
        
        for page_num, image in enumerate(images, 1):
            print(f"Processing page {page_num}/{len(images)}...")
            
            # Convert PIL image to OpenCV format
            opencv_image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
            
            # Preprocess image
            processed_image = self.preprocess_image(opencv_image)
            
            # Try different OCR configurations for better results
            ocr_configs = [
                '--oem 3 --psm 6',  # Assume uniform text block
                '--oem 3 --psm 4',  # Assume single column of text
                '--oem 3 --psm 11'  # Sparse text
            ]
            
            page_text = ""
            for config in ocr_configs:
                try:
                    text = pytesseract.image_to_string(processed_image, config=config)
                    if len(text) > len(page_text):
                        page_text = text
                except:
                    continue
            
            # Extract account information from this page
            page_account = self.extract_account_info(page_text)
            if page_account:
                page_accounts.append((page_num, page_account))
            
            all_text.append({
                'page_num': page_num,
                'text': page_text,
                'account': page_account
            })
        
        return all_text

    def extract_account_info(self, text):
        """
        Extract account information from text
        """
        lines = text.split('\n')
        for line in lines:
            for pattern in self.account_patterns:
                match = re.search(pattern, line, re.IGNORECASE)
                if match:
                    return match.group(0).strip()
        return None

    def parse_table_data(self, text, current_account):
        """
        Parse table data from OCR text
        """
        rows = []
        lines = text.split('\n')
        
        # Regular expressions for different data types
        date_pattern = r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}'
        currency_pattern = r'(USD|ZAR|EUR|GBP)'
        amount_pattern = r'[\d,]+\.\d{2}'
        
        current_row = {}
        header_found = False
        header_columns = []
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # Try to identify header row
            if not header_found and 'DATE' in line.upper() and any(col in line.upper() for col in ['SOURCE', 'DESCRIPTION', 'DEBIT']):
                header_found = True
                header_columns = re.split(r'\s{2,}', line)
                continue
            
            # Skip if line is too short or likely not a data row
            if len(line.split()) < 3:
                continue
            
            # Try to parse as data row
            parts = re.split(r'\s{2,}', line)
            
            # If we have enough parts, treat as data row
            if len(parts) >= 3:
                row_data = {
                    'ACCOUNT': current_account,
                    'DATE': '',
                    'SOURCE': '',
                    'DESCRIPTION': '',
                    'REFERENCE': '',
                    'CURRENCY': '',
                    'DEBIT(SOURCE)': '',
                    'CREDIT(SOURCE)': '',
                    'DEBIT(USD)': '',
                    'CREDIT(USD)': '',
                    'RUNNING BALANCE(USD)': '',
                    'RELATED ACCOUNT': '',
                    'RAW_TEXT': line  # Keep raw text for debugging
                }
                
                # Try to extract date
                date_match = re.search(date_pattern, line)
                if date_match:
                    row_data['DATE'] = date_match.group(0)
                
                # Try to extract currency
                currency_match = re.search(currency_pattern, line)
                if currency_match:
                    row_data['CURRENCY'] = currency_match.group(0)
                
                # Try to extract amounts (debit/credit/balance)
                amounts = re.findall(amount_pattern, line)
                if amounts:
                    # Distribute amounts based on typical order
                    amount_idx = 0
                    for field in ['DEBIT(SOURCE)', 'CREDIT(SOURCE)', 'DEBIT(USD)', 'CREDIT(USD)', 'RUNNING BALANCE(USD)']:
                        if amount_idx < len(amounts):
                            row_data[field] = amounts[amount_idx]
                            amount_idx += 1
                
                # Assign remaining parts to description/source
                text_parts = [p for p in parts if not re.search(amount_pattern, p) and not re.search(date_pattern, p)]
                if text_parts:
                    row_data['DESCRIPTION'] = ' '.join(text_parts)
                
                rows.append(row_data)
        
        return rows

    def process_scanned_pdf(self, pdf_path, output_excel_path, use_google_vision=False):
        """
        Main function to process scanned PDF and export to Excel
        """
        print(f"Processing scanned PDF: {pdf_path}")
        
        # Extract text using OCR
        pages_data = self.extract_text_from_scanned_pdf(pdf_path)
        
        all_rows = []
        current_account = None
        
        for page_data in pages_data:
            page_num = page_data['page_num']
            text = page_data['text']
            page_account = page_data['account']
            
            # Update current account if new account found
            if page_account:
                current_account = page_account
                print(f"Page {page_num}: Found account - {current_account}")
            
            # Parse table data from this page
            page_rows = self.parse_table_data(text, current_account)
            
            if page_rows:
                print(f"Page {page_num}: Extracted {len(page_rows)} rows")
                all_rows.extend(page_rows)
        
        # Create DataFrame
        if all_rows:
            df = pd.DataFrame(all_rows)
            
            # Remove rows with no meaningful data
            df = df[~df['RAW_TEXT'].str.contains('Page|Account|Balance|Total', case=False, na=False)]
            
            # Drop the RAW_TEXT column
            if 'RAW_TEXT' in df.columns:
                df = df.drop('RAW_TEXT', axis=1)
            
            # Reorder columns
            column_order = ['ACCOUNT'] + [col for col in self.expected_columns if col in df.columns]
            df = df[column_order]
            
            # Export to Excel
            print(f"Exporting to Excel: {output_excel_path}")
            with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
                df.to_excel(writer, sheet_name='Financial Data', index=False)
                
                # Auto-adjust column widths
                worksheet = writer.sheets['Financial Data']
                for column in worksheet.columns:
                    max_length = 0
                    column_letter = column[0].column_letter
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    adjusted_width = min(max_length + 2, 50)
                    worksheet.column_dimensions[column_letter].width = adjusted_width
                
                # Create a summary sheet
                account_summary = df['ACCOUNT'].value_counts().reset_index()
                account_summary.columns = ['Account', 'Number of Transactions']
                account_summary.to_excel(writer, sheet_name='Account Summary', index=False)
            
            print(f"Successfully exported {len(df)} rows to {output_excel_path}")
            return df
        else:
            print("No data extracted from PDF")
            return pd.DataFrame()

def main():
    """
    Main function to run the script
    """
    print("=" * 60)
    print("Scanned PDF to Excel Converter for Financial Statements")
    print("=" * 60)
    
    # Get input file path
    pdf_path = input("\nEnter the path to your scanned PDF file: ").strip()
    
    if not pdf_path:
        pdf_path = "input.pdf"
        print(f"Using default: {pdf_path}")
    
    # Check if file exists
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        print(f"Error: File {pdf_path} not found!")
        return
    
    # Ask for Tesseract path (Windows users might need this)
    tesseract_path = input("\nEnter Tesseract path (press Enter if not needed): ").strip()
    if not tesseract_path:
        tesseract_path = None
    
    # Generate output filename
    output_excel_path = pdf_file.stem + "_extracted.xlsx"
    
    try:
        # Initialize processor
        processor = ScannedPDFToExcel(tesseract_path)
        
        # Process the PDF
        df = processor.process_scanned_pdf(pdf_path, output_excel_path)
        
        if not df.empty:
            print("\n" + "=" * 60)
            print("EXTRACTION SUMMARY")
            print("=" * 60)
            print(f"\nTotal rows extracted: {len(df)}")
            print(f"\nAccounts found:")
            for account in df['ACCOUNT'].unique():
                count = len(df[df['ACCOUNT'] == account])
                print(f"  - {account}: {count} transactions")
            
            print(f"\nColumns exported: {', '.join(df.columns.tolist())}")
            print(f"\n✅ File saved as: {output_excel_path}")
            
            # Show preview
            print("\n" + "=" * 60)
            print("PREVIEW (First 5 rows)")
            print("=" * 60)
            print(df.head(5).to_string())
            
        else:
            print("\n❌ No data was extracted. Please check:")
            print("   1. The PDF contains readable text")
            print("   2. Tesseract OCR is properly installed")
            print("   3. Image quality is sufficient")
            
    except Exception as e:
        print(f"\n❌ Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    # Install required packages if not already installed
    import subprocess
    import sys
    
    required_packages = [
        'pandas',
        'pytesseract',
        'pdf2image',
        'opencv-python',
        'openpyxl',
        'numpy'
    ]
    
    print("Checking and installing required packages...")
    for package in required_packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    
    # Check for poppler (required for pdf2image)
    try:
        from pdf2image import convert_from_path
        # Test if poppler is available
        convert_from_path('dummy.pdf', dpi=1)
    except Exception as e:
        print("\n⚠️  Warning: pdf2image requires poppler-utils")
        print("   - Windows: Download from: http://blog.alivate.com.au/poppler-windows/")
        print("   - Mac: brew install poppler")
        print("   - Linux: sudo apt-get install poppler-utils")
    
    # Check for Tesseract OCR
    try:
        pytesseract.get_tesseract_version()
    except:
        print("\n⚠️  Warning: Tesseract OCR not found")
        print("   Please install Tesseract OCR:")
        print("   - Windows: Download from: https://github.com/UB-Mannheim/tesseract/wiki")
        print("   - Mac: brew install tesseract")
        print("   - Linux: sudo apt-get install tesseract-ocr")
    
    main()

ModuleNotFoundError: No module named 'pytesseract'

In [3]:
python.exe -m pip install --upgrade pip

SyntaxError: invalid syntax (842801469.py, line 1)

In [4]:
!python.exe -m pip install --upgrade pip

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\python311\\lib\\site-packages\\pip-24.0.dist-info\\AUTHORS.txt'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Create an empty list to store numbers that meet the given conditions
nl = []

# Iterate through numbers from 1500 to 2700 (inclusive)
for x in range(1500, 2701):
    # Check if the number is divisible by 7 and 5 without any remainder
    if (x % 7 == 0) and (x % 5 == 0):
        # If the conditions are met, convert the number to a string and append it to the list
        nl.append(str(x))

# Join the numbers in the list with a comma and print the result
print(','.join(nl))

1505,1540,1575,1610,1645,1680,1715,1750,1785,1820,1855,1890,1925,1960,1995,2030,2065,2100,2135,2170,2205,2240,2275,2310,2345,2380,2415,2450,2485,2520,2555,2590,2625,2660,2695


In [2]:
n = 4
src = 'A'
aux = 'C'
des = 'B'

total_moves = 2 ** n - 1
if n % 2 == 0:
    des, aux = aux, des

rods = { 'A': list(range(n, 0, -1)), 'B': [], 'C': [] }

for move in range(1, total_moves + 1):
    if move % 3 == 1:
        from_rod, to_rod = ('A', 'B') if (rods['A'] and (not rods['B'] or rods['A'][-1] < rods['B'][-1])) else ('B', 'A')
    elif move % 3 == 2:
        from_rod, to_rod = ('A', 'C') if (rods['A'] and (not rods['C'] or rods['A'][-1] < rods['C'][-1])) else ('C', 'A')
    else:
        from_rod, to_rod = ('B', 'C') if (rods['B'] and (not rods['C'] or rods['B'][-1] < rods['C'][-1])) else ('C', 'B')
    
    disk = rods[from_rod].pop()
    rods[to_rod].append(disk)
    print(f"Move disk {disk} from {from_rod} to {to_rod}")

Move disk 1 from A to B
Move disk 2 from A to C
Move disk 1 from B to C
Move disk 3 from A to B
Move disk 1 from C to A
Move disk 2 from C to B
Move disk 1 from A to B
Move disk 4 from A to C
Move disk 1 from B to C
Move disk 2 from B to A
Move disk 1 from C to A
Move disk 3 from B to C
Move disk 1 from A to B
Move disk 2 from A to C
Move disk 1 from B to C


In [3]:
n = 4
src = 'A'
des = 'B'
aux = 'C'

def move_disks(n, src, des, aux):
    if n == 1:
        print(f"Move disk 1 from {src} to {des}")
        return
    move_disks(n - 1, src, aux, des)
    print(f"Move disk {n} from {src} to {des}")
    move_disks(n - 1, aux, des, src)

move_disks(n, src, des, aux)

Move disk 1 from A to C
Move disk 2 from A to B
Move disk 1 from C to B
Move disk 3 from A to C
Move disk 1 from B to A
Move disk 2 from B to C
Move disk 1 from A to C
Move disk 4 from A to B
Move disk 1 from C to B
Move disk 2 from C to A
Move disk 1 from B to A
Move disk 3 from C to B
Move disk 1 from A to C
Move disk 2 from A to B
Move disk 1 from C to B


In [4]:
def tower_of_hanoi_iterative(n, src='A', aux='C', des='B'):
    """
    Iterative solution for the Tower of Hanoi puzzle.
    
    Args:
        n (int): Number of disks
        src (str): Source rod identifier
        aux (str): Auxiliary rod identifier
        des (str): Destination rod identifier
    """
    # Calculate total moves required
    total_moves = 2 ** n - 1
    
    # For even number of disks, swap auxiliary and destination roles
    if n % 2 == 0:
        des, aux = aux, des
    
    # Represent rods as stacks (lists) with disks (largest at bottom)
    rods = {src: list(range(n, 0, -1)), aux: [], des: []}
    
    # Iterate through each move
    for move in range(1, total_moves + 1):
        # Determine which rods to move between based on move number modulo 3
        if move % 3 == 1:
            from_rod, to_rod = (src, des) if (rods[src] and (not rods[des] or rods[src][-1] < rods[des][-1])) else (des, src)
        elif move % 3 == 2:
            from_rod, to_rod = (src, aux) if (rods[src] and (not rods[aux] or rods[src][-1] < rods[aux][-1])) else (aux, src)
        else:  # move % 3 == 0
            from_rod, to_rod = (aux, des) if (rods[aux] and (not rods[des] or rods[aux][-1] < rods[des][-1])) else (des, aux)
        
        # Perform the valid move
        disk = rods[from_rod].pop()
        rods[to_rod].append(disk)
        print(f"Move disk {disk} from {from_rod} to {to_rod}")

# Example usage for 3 disks
print("Iterative solution for 3 disks:")
tower_of_hanoi_iterative(3, 'A', 'C', 'B')

Iterative solution for 3 disks:
Move disk 1 from A to B
Move disk 2 from A to C
Move disk 1 from B to C
Move disk 3 from A to B
Move disk 1 from C to A
Move disk 2 from C to B
Move disk 1 from A to B


In [5]:
import time
import sys

# Increase recursion limit for larger n values
sys.setrecursionlimit(3000)

# ==================== RECURSIVE VERSION ====================
def tower_of_hanoi_recursive(n, src, des, aux, moves_counter):
    """Recursive Tower of Hanoi that counts moves without printing"""
    if n == 1:
        moves_counter[0] += 1
        return
    tower_of_hanoi_recursive(n - 1, src, aux, des, moves_counter)
    moves_counter[0] += 1
    tower_of_hanoi_recursive(n - 1, aux, des, src, moves_counter)

# ==================== ITERATIVE VERSION ====================
def tower_of_hanoi_iterative(n, src='A', aux='C', des='B'):
    """Iterative Tower of Hanoi that counts moves without printing"""
    total_moves = 2 ** n - 1
    moves_count = 0
    
    # For even number of disks, swap auxiliary and destination
    if n % 2 == 0:
        des, aux = aux, des
    
    # Represent rods as stacks
    rods = {src: list(range(n, 0, -1)), aux: [], des: []}
    
    # Perform all moves
    for move in range(1, total_moves + 1):
        moves_count += 1
        
        # Determine which rods to move between
        if move % 3 == 1:
            from_rod, to_rod = (src, des) if (rods[src] and (not rods[des] or rods[src][-1] < rods[des][-1])) else (des, src)
        elif move % 3 == 2:
            from_rod, to_rod = (src, aux) if (rods[src] and (not rods[aux] or rods[src][-1] < rods[aux][-1])) else (aux, src)
        else:
            from_rod, to_rod = (aux, des) if (rods[aux] and (not rods[des] or rods[aux][-1] < rods[des][-1])) else (des, aux)
        
        # Execute the move
        disk = rods[from_rod].pop()
        rods[to_rod].append(disk)
    
    return moves_count

# ==================== TIMING FUNCTION ====================
def time_tower_of_hanoi(n_values):
    """Time both recursive and iterative versions for different n values"""
    
    print("=" * 70)
    print(f"{'n':<6} {'Recursive Time (s)':<20} {'Iterative Time (s)':<20} {'Moves':<10}")
    print("=" * 70)
    
    for n in n_values:
        # Time recursive version
        moves_counter = [0]
        start_time = time.perf_counter()
        tower_of_hanoi_recursive(n, 'A', 'B', 'C', moves_counter)
        recursive_time = time.perf_counter() - start_time
        
        # Time iterative version
        start_time = time.perf_counter()
        moves = tower_of_hanoi_iterative(n)
        iterative_time = time.perf_counter() - start_time
        
        # Verify both versions made same number of moves
        assert moves == moves_counter[0], f"Move count mismatch: recursive={moves_counter[0]}, iterative={moves}"
        
        print(f"{n:<6} {recursive_time:<20.6f} {iterative_time:<20.6f} {moves:<10}")

# ==================== RUN TIMING TESTS ====================
if __name__ == "__main__":
    # Test with different numbers of disks
    test_values = [5, 10, 15, 20, 22, 24]
    
    print("Timing Tower of Hanoi Algorithms\n")
    print("Note: Printing is disabled to accurately measure computation time.\n")
    
    time_tower_of_hanoi(test_values)
    
    # Optional: Detailed timing for a single value
    print("\n" + "=" * 70)
    print("DETAILED TIMING FOR N=20")
    print("=" * 70)
    
    n = 20
    
    # Multiple runs for average
    recursive_runs = []
    iterative_runs = []
    
    for i in range(5):
        moves_counter = [0]
        start = time.perf_counter()
        tower_of_hanoi_recursive(n, 'A', 'B', 'C', moves_counter)
        recursive_runs.append(time.perf_counter() - start)
        
        start = time.perf_counter()
        moves = tower_of_hanoi_iterative(n)
        iterative_runs.append(time.perf_counter() - start)
    
    print(f"Recursive - Min: {min(recursive_runs):.6f}s, Max: {max(recursive_runs):.6f}s, Avg: {sum(recursive_runs)/5:.6f}s")
    print(f"Iterative - Min: {min(iterative_runs):.6f}s, Max: {max(iterative_runs):.6f}s, Avg: {sum(iterative_runs)/5:.6f}s")
    print(f"Total moves for n={n}: {2**n - 1}")

Timing Tower of Hanoi Algorithms

Note: Printing is disabled to accurately measure computation time.

n      Recursive Time (s)   Iterative Time (s)   Moves     
5      0.000023             0.000072             31        
10     0.000320             0.000671             1023      
15     0.009927             0.017432             32767     
20     0.124898             0.358821             1048575   
22     0.571165             1.404596             4194303   
24     2.226868             5.428612             16777215  

DETAILED TIMING FOR N=20
Recursive - Min: 0.129218s, Max: 0.149471s, Avg: 0.141385s
Iterative - Min: 0.342035s, Max: 0.394582s, Avg: 0.366134s
Total moves for n=20: 1048575


In [6]:
def simple_timing_test(n):
    """Simple timing test for a single n value"""
    
    print(f"\n--- Timing Test for {n} Disks ---")
    
    # Recursive timing
    moves_counter = [0]
    start = time.perf_counter()
    tower_of_hanoi_recursive(n, 'A', 'B', 'C', moves_counter)
    rec_time = time.perf_counter() - start
    
    # Iterative timing
    start = time.perf_counter()
    moves = tower_of_hanoi_iterative(n)
    it_time = time.perf_counter() - start
    
    print(f"Recursive time: {rec_time:.6f} seconds")
    print(f"Iterative time: {it_time:.6f} seconds")
    print(f"Speedup: {rec_time/it_time:.2f}x")
    print(f"Total moves: {moves}")
    print(f"Theoretical moves: {2**n - 1}")
    
    return rec_time, it_time

# Test with different n values
for n in [10, 15, 20]:
    simple_timing_test(n)


--- Timing Test for 10 Disks ---
Recursive time: 0.000312 seconds
Iterative time: 0.001217 seconds
Speedup: 0.26x
Total moves: 1023
Theoretical moves: 1023

--- Timing Test for 15 Disks ---
Recursive time: 0.012540 seconds
Iterative time: 0.026324 seconds
Speedup: 0.48x
Total moves: 32767
Theoretical moves: 32767

--- Timing Test for 20 Disks ---
Recursive time: 0.217851 seconds
Iterative time: 0.354401 seconds
Speedup: 0.61x
Total moves: 1048575
Theoretical moves: 1048575


In [3]:
import subprocess
import sys
import importlib.util

# Function to install missing packages
def install_and_import(package):
    try:
        importlib.import_module(package.replace('-', '_'))
        print(f"✓ {package} already installed")
        return True
    except ImportError:
        print(f"Installing {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"✓ Successfully installed {package}")
            return True
        except Exception as e:
            print(f"✗ Failed to install {package}: {e}")
            return False

# Install required packages
required_packages = ['tabula-py', 'pandas', 'pdfplumber', 'openpyxl']
for package in required_packages:
    install_and_import(package)

# Now import the packages
import pandas as pd
import pdfplumber
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# Try to import tabula, but handle if it fails
try:
    import tabula
    TABULA_AVAILABLE = True
    print("✓ Tabula imported successfully")
except ImportError:
    TABULA_AVAILABLE = False
    print("⚠ Tabula not available, will use pdfplumber only")

def extract_tables_from_pdf(pdf_path):
    """
    Extract tables from PDF using multiple methods for better accuracy
    """
    all_data = []
    
    # Method 1: Using tabula (good for tabular data) - only if available
    if TABULA_AVAILABLE:
        try:
            tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True, 
                                     lattice=True, pandas_options={'header': None})
            for table in tables:
                if not table.empty:
                    all_data.append(table)
            print(f"Tabula extracted {len(tables)} tables")
        except Exception as e:
            print(f"Tabula extraction warning: {e}")
    else:
        print("Skipping tabula extraction (module not available)")
    
    # Method 2: Using pdfplumber (good for text extraction)
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    # Look for account information in the text
                    lines = text.split('\n')
                    for line in lines:
                        if any(account_type in line for account_type in 
                              ['Compensation', 'Insurance Proceeds', 'Zim Account', 'Account']):
                            all_data.append({'account_info': line, 'page': page_num + 1})
                
                # Try to extract tables using pdfplumber
                tables = page.extract_tables()
                for table in tables:
                    if table and len(table) > 1:  # Has at least header and one row
                        df = pd.DataFrame(table[1:], columns=table[0])
                        if not df.empty:
                            all_data.append(df)
    except Exception as e:
        print(f"pdfplumber extraction warning: {e}")
    
    return all_data

def clean_dataframe(df):
    """
    Clean and structure the extracted data
    """
    # If the dataframe is empty or has no columns, return empty dataframe
    if df.empty or len(df.columns) == 0:
        return pd.DataFrame()
    
    # Convert all columns to string for processing
    df = df.astype(str)
    
    # Define expected columns
    expected_columns = [
        'DATE', 'SOURCE', 'DESCRIPTION', 'REFERENCE', 'CURRENCY',
        'DEBIT(SOURCE)', 'CREDIT(SOURCE)', 'DEBIT(USD)', 'CREDIT(USD)',
        'RUNNING BALANCE(USD)', 'RELATED ACCOUNT'
    ]
    
    # If dataframe has more columns than expected, try to identify the correct ones
    if len(df.columns) > len(expected_columns):
        # Look for rows that might contain column headers
        for idx, row in df.iterrows():
            row_str = ' '.join(row.astype(str))
            if 'DATE' in row_str.upper() and 'SOURCE' in row_str.upper():
                # This row contains headers, use it to set column names
                headers = [col.strip() for col in row if col and col != 'nan']
                if len(headers) >= len(expected_columns):
                    df.columns = headers[:len(expected_columns)]
                    df = df.iloc[idx+1:].reset_index(drop=True)
                    break
    
    # Ensure correct number of columns
    if len(df.columns) < len(expected_columns):
        # Add missing columns
        for i in range(len(df.columns), len(expected_columns)):
            df[f'col_{i}'] = ''
        df.columns = expected_columns[:len(df.columns)] + expected_columns[len(df.columns):]
    elif len(df.columns) > len(expected_columns):
        df = df.iloc[:, :len(expected_columns)]
        df.columns = expected_columns
    
    return df

def extract_account_info(text):
    """
    Extract account information from text
    """
    account_patterns = [
        r'(Compensation\s*\(\s*Zim\s*\)\s*Account)',
        r'(Insurance\s*Proceeds\s*Zim\s*Account)',
        r'([A-Za-z\s]+Zim\s*Account)',
        r'([A-Za-z\s]+Account)'
    ]
    
    for pattern in account_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return 'Unknown Account'

def process_pdf_to_excel(pdf_path, output_excel_path):
    """
    Main function to process PDF and export to Excel
    """
    print(f"\nProcessing PDF: {pdf_path}")
    
    # Extract data from PDF
    extracted_data = extract_tables_from_pdf(pdf_path)
    
    # Process extracted data
    all_dfs = []
    current_account = 'Unknown Account'
    
    for data in extracted_data:
        if isinstance(data, dict) and 'account_info' in data:
            # This is account information
            current_account = extract_account_info(data['account_info'])
            print(f"Found account: {current_account} (Page {data.get('page', 'Unknown')})")
        elif isinstance(data, pd.DataFrame):
            # This is tabular data
            df = clean_dataframe(data)
            if not df.empty:
                # Add account information to each row
                df['ACCOUNT'] = current_account
                all_dfs.append(df)
                print(f"Added table with {len(df)} rows for account: {current_account}")
    
    # Combine all dataframes
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        # Remove any completely empty rows
        final_df = final_df.dropna(how='all')
        
        # Reorder columns to put ACCOUNT at the beginning
        if 'ACCOUNT' in final_df.columns:
            cols = final_df.columns.tolist()
            cols.insert(0, cols.pop(cols.index('ACCOUNT')))
            final_df = final_df[cols]
        
        # Export to Excel
        print(f"\nExporting to Excel: {output_excel_path}")
        try:
            with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
                final_df.to_excel(writer, sheet_name='Financial Data', index=False)
                
                # Auto-adjust column widths
                worksheet = writer.sheets['Financial Data']
                for column in worksheet.columns:
                    max_length = 0
                    column_letter = column[0].column_letter
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    adjusted_width = min(max_length + 2, 50)
                    worksheet.column_dimensions[column_letter].width = adjusted_width
            
            print(f"✅ Successfully exported {len(final_df)} rows to {output_excel_path}")
            return final_df
        except Exception as e:
            print(f"❌ Error exporting to Excel: {e}")
            # Fallback to CSV if Excel export fails
            csv_path = output_excel_path.replace('.xlsx', '.csv')
            final_df.to_csv(csv_path, index=False)
            print(f"✅ Data saved to CSV instead: {csv_path}")
            return final_df
    else:
        print("❌ No data extracted from PDF")
        return pd.DataFrame()

def main():
    """
    Main function to run the script
    """
    print("=" * 60)
    print("PDF to Excel Financial Data Extractor")
    print("=" * 60)
    
    # Your specific file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    print(f"\nProcessing file: {pdf_path}")
    
    # Check if input file exists
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        print(f"\n❌ Error: File not found at:")
        print(f"   {pdf_path}")
        print("\nPlease check:")
        print("1. The file path is correct")
        print("2. The file exists at that location")
        print("3. You have permission to read the file")
        return
    
    # Generate output Excel filename (save in the same directory as the PDF)
    output_excel_path = pdf_file.parent / (pdf_file.stem + "_extracted.xlsx")
    
    try:
        # Process the PDF
        print(f"\nStarting extraction...")
        df = process_pdf_to_excel(str(pdf_path), str(output_excel_path))
        
        if not df.empty:
            print("\n" + "=" * 60)
            print("✅ EXTRACTION SUMMARY")
            print("=" * 60)
            print(f"Total rows extracted: {len(df)}")
            print(f"Total columns: {len(df.columns)}")
            print(f"Columns: {', '.join(df.columns.tolist())}")
            
            print("\n📊 Preview of first 10 rows:")
            print(df.head(10))
            
            # Show unique accounts found
            if 'ACCOUNT' in df.columns:
                unique_accounts = df['ACCOUNT'].unique()
                print(f"\n📁 Accounts found: {', '.join(unique_accounts)}")
            
            print(f"\n📁 Output saved to: {output_excel_path}")
        else:
            print("\n❌ No data was extracted. Please check the PDF format.")
            print("The PDF might be:")
            print("- Scanned/image-based (needs OCR)")
            print("- Password protected")
            print("- In a format that doesn't contain extractable tables")
            
    except Exception as e:
        print(f"\n❌ Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Installing tabula-py...
✗ Failed to install tabula-py: Command '['c:\\Python311\\python.exe', '-m', 'pip', 'install', 'tabula-py']' returned non-zero exit status 1.
✓ pandas already installed
Installing pdfplumber...
✗ Failed to install pdfplumber: Command '['c:\\Python311\\python.exe', '-m', 'pip', 'install', 'pdfplumber']' returned non-zero exit status 1.
✓ openpyxl already installed


ModuleNotFoundError: No module named 'pdfplumber'

In [4]:
import PyPDF2
import pandas as pd
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

def install_pypdf2():
    """Install PyPDF2 if not available"""
    import subprocess
    import sys
    
    try:
        import PyPDF2
        print("✓ PyPDF2 already installed")
        return True
    except ImportError:
        print("Installing PyPDF2...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])
            print("✓ Successfully installed PyPDF2")
            return True
        except Exception as e:
            print(f"✗ Failed to install PyPDF2: {e}")
            return False

# Install PyPDF2 if needed
if not install_pypdf2():
    print("Failed to install required package. Please run: pip install PyPDF2")
    exit(1)

import PyPDF2

def extract_text_from_pdf(pdf_path):
    """
    Extract text from PDF using PyPDF2
    """
    all_text = []
    
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            print(f"PDF has {len(pdf_reader.pages)} pages")
            
            for page_num, page in enumerate(pdf_reader.pages):
                text = page.extract_text()
                if text:
                    all_text.append({
                        'page': page_num + 1,
                        'text': text
                    })
                    print(f"✓ Extracted text from page {page_num + 1}")
                else:
                    print(f"⚠ No text found on page {page_num + 1}")
    
    except Exception as e:
        print(f"Error reading PDF: {e}")
    
    return all_text

def parse_financial_data(text_data):
    """
    Parse financial data from extracted text
    """
    all_rows = []
    current_account = "Unknown"
    
    for page_data in text_data:
        page_num = page_data['page']
        text = page_data['text']
        
        # Split into lines
        lines = text.split('\n')
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # Look for account headers
            account_match = re.search(r'(Compensation.*Account|Insurance.*Proceeds.*Account|.*Zim.*Account|.*Account)', line, re.IGNORECASE)
            if account_match and len(line) < 100:  # Account headers are usually short
                current_account = account_match.group(1).strip()
                print(f"Found account on page {page_num}: {current_account}")
                continue
            
            # Try to parse as transaction line
            # Look for patterns that might indicate financial data
            if re.search(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', line):  # Contains date
                transaction = parse_transaction_line(line, current_account, page_num)
                if transaction:
                    all_rows.append(transaction)
            elif re.search(r'[\d,]+\.\d{2}', line) and any(word in line.lower() for word in ['debit', 'credit', 'balance', 'usd', 'zwl']):
                # Line contains currency amounts and keywords
                transaction = parse_transaction_line(line, current_account, page_num)
                if transaction:
                    all_rows.append(transaction)
    
    return all_rows

def parse_transaction_line(line, account, page_num):
    """
    Parse a single transaction line
    """
    # Initialize transaction dictionary
    transaction = {
        'ACCOUNT': account,
        'PAGE': page_num,
        'RAW_TEXT': line,
        'DATE': '',
        'DESCRIPTION': '',
        'DEBIT': '',
        'CREDIT': '',
        'BALANCE': '',
        'CURRENCY': ''
    }
    
    # Try to extract date
    date_match = re.search(r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})', line)
    if date_match:
        transaction['DATE'] = date_match.group(1)
    
    # Try to extract currency amounts
    amounts = re.findall(r'([\d,]+\.\d{2})', line)
    
    # Try to identify currency
    if 'USD' in line.upper():
        transaction['CURRENCY'] = 'USD'
    elif 'ZWL' in line.upper() or 'ZIM' in line.upper():
        transaction['CURRENCY'] = 'ZWL'
    
    # Try to identify if it's debit or credit
    if 'DEBIT' in line.upper() and amounts:
        transaction['DEBIT'] = amounts[0]
    elif 'CREDIT' in line.upper() and amounts:
        transaction['CREDIT'] = amounts[0]
    elif 'BALANCE' in line.upper() and amounts:
        transaction['BALANCE'] = amounts[-1]
    elif len(amounts) >= 2:
        # Assume first amount is debit/credit, last is balance
        transaction['DEBIT'] = amounts[0]
        transaction['BALANCE'] = amounts[-1]
    elif len(amounts) == 1:
        transaction['BALANCE'] = amounts[0]
    
    # Extract description (everything after date and before amounts)
    if transaction['DATE']:
        parts = line.split(transaction['DATE'])
        if len(parts) > 1:
            desc = parts[1].strip()
            # Remove amounts from description
            for amt in amounts:
                desc = desc.replace(amt, '')
            transaction['DESCRIPTION'] = desc.strip()
    
    return transaction

def create_dataframe(transactions):
    """
    Create pandas DataFrame from parsed transactions
    """
    if not transactions:
        return pd.DataFrame()
    
    df = pd.DataFrame(transactions)
    
    # Reorder columns
    column_order = ['ACCOUNT', 'DATE', 'DESCRIPTION', 'DEBIT', 'CREDIT', 
                   'BALANCE', 'CURRENCY', 'PAGE', 'RAW_TEXT']
    
    # Only keep columns that exist
    existing_cols = [col for col in column_order if col in df.columns]
    df = df[existing_cols]
    
    return df

def save_to_excel(df, output_path):
    """
    Save DataFrame to Excel with formatting
    """
    if df.empty:
        print("No data to save")
        return False
    
    try:
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Financial Data', index=False)
            
            # Auto-adjust column widths
            worksheet = writer.sheets['Financial Data']
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter
                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width
        
        print(f"✅ Successfully saved to {output_path}")
        return True
    
    except Exception as e:
        print(f"❌ Error saving to Excel: {e}")
        # Fallback to CSV
        csv_path = output_path.replace('.xlsx', '.csv')
        df.to_csv(csv_path, index=False)
        print(f"✅ Saved to CSV instead: {csv_path}")
        return True

def main():
    """
    Main function to run the script
    """
    print("=" * 60)
    print("PDF to Excel Financial Data Extractor")
    print("(Using PyPDF2 - No tabula or pdfplumber required)")
    print("=" * 60)
    
    # Your specific file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    print(f"\n📁 Processing file: {pdf_path}")
    
    # Check if input file exists
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        print(f"\n❌ Error: File not found at:")
        print(f"   {pdf_path}")
        print("\nPlease check:")
        print("1. The file path is correct")
        print("2. The file exists at that location")
        print("3. You have permission to read the file")
        return
    
    # Generate output filename
    output_excel_path = pdf_file.parent / (pdf_file.stem + "_extracted.xlsx")
    
    try:
        # Step 1: Extract text from PDF
        print("\n📖 Extracting text from PDF...")
        text_data = extract_text_from_pdf(pdf_path)
        
        if not text_data:
            print("❌ No text could be extracted from the PDF")
            return
        
        # Step 2: Parse financial data
        print("\n🔍 Parsing financial data...")
        transactions = parse_financial_data(text_data)
        
        if not transactions:
            print("❌ No financial transactions found in the text")
            print("\nThe extracted text preview:")
            for page in text_data[:2]:  # Show first 2 pages preview
                print(f"\nPage {page['page']} (first 500 chars):")
                print(page['text'][:500])
            return
        
        # Step 3: Create DataFrame
        print(f"\n📊 Creating DataFrame with {len(transactions)} potential transactions...")
        df = create_dataframe(transactions)
        
        # Step 4: Save to Excel
        print("\n💾 Saving to Excel...")
        save_to_excel(df, output_excel_path)
        
        # Step 5: Show summary
        print("\n" + "=" * 60)
        print("✅ EXTRACTION SUMMARY")
        print("=" * 60)
        print(f"Total transactions extracted: {len(df)}")
        print(f"Output file: {output_excel_path}")
        
        if not df.empty:
            print("\n📊 Preview of first 10 rows:")
            print(df[['ACCOUNT', 'DATE', 'DESCRIPTION', 'DEBIT', 'CREDIT', 'BALANCE']].head(10))
            
            # Show unique accounts
            if 'ACCOUNT' in df.columns:
                unique_accounts = df['ACCOUNT'].unique()
                print(f"\n📁 Accounts found: {', '.join(unique_accounts)}")
    
    except Exception as e:
        print(f"\n❌ Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'PyPDF2'

In [1]:
import pandas as pd

df = pd.read_csv('data.csv')

print(df.head(10))

KeyboardInterrupt: 

In [5]:
import re
import os
from pathlib import Path
import csv
from datetime import datetime

def extract_text_from_pdf_simple(pdf_path):
    """
    Extract text from PDF using basic binary reading
    This is a simplified approach that doesn't require any external libraries
    """
    text_content = []
    
    try:
        with open(pdf_path, 'rb') as file:
            pdf_data = file.read()
            
        # Decode as latin-1 to handle binary data
        pdf_text = pdf_data.decode('latin-1', errors='ignore')
        
        # Split into pages (PDF pages are separated by 'endobj' or similar markers)
        pages = re.split(r'endobj\s*\d+\s*obj', pdf_text)
        
        for page_num, page in enumerate(pages):
            # Extract text content between BT and ET operators (text blocks in PDF)
            text_blocks = re.findall(r'BT(.*?)ET', page, re.DOTALL)
            
            page_text = ""
            for block in text_blocks:
                # Extract text between parentheses
                text_parts = re.findall(r'\((.*?)\)', block)
                if text_parts:
                    # Join text parts and clean up
                    clean_text = ' '.join(text_parts)
                    # Fix common PDF encoding issues
                    clean_text = clean_text.replace('\\', '')
                    page_text += clean_text + ' '
            
            if page_text.strip():
                text_content.append({
                    'page': page_num + 1,
                    'text': page_text.strip()
                })
                print(f"✓ Extracted text from page {page_num + 1}")
    
    except Exception as e:
        print(f"Error reading PDF: {e}")
    
    return text_content

def extract_text_alternative(pdf_path):
    """
    Alternative method using string searching
    """
    text_content = []
    
    try:
        with open(pdf_path, 'rb') as file:
            content = file.read().decode('latin-1', errors='ignore')
        
        # Find all text between stream and endstream markers
        stream_pattern = r'stream(.*?)endstream'
        streams = re.findall(stream_pattern, content, re.DOTALL)
        
        for stream_num, stream in enumerate(streams):
            # Try to find readable text (letters, numbers, spaces, basic punctuation)
            readable = re.findall(r'[A-Za-z0-9\s\.,:;\-\$\%\(\)]+', stream)
            if readable:
                combined_text = ' '.join(readable)
                if len(combined_text.strip()) > 50:  # Only keep substantial text
                    text_content.append({
                        'page': stream_num // 10 + 1,  # Approximate page number
                        'text': combined_text.strip()
                    })
                    print(f"✓ Extracted block {stream_num + 1}")
    
    except Exception as e:
        print(f"Error in alternative method: {e}")
    
    return text_content

def parse_transactions(text):
    """
    Parse transaction data from extracted text
    """
    transactions = []
    
    # Common patterns in financial documents
    patterns = {
        'date': r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b',
        'amount': r'\b[\d,]+\.\d{2}\b',
        'account': r'(Compensation|Insurance|Zim|USD|ZWL).*?(?:Account)?',
        'reference': r'(?:Ref|Reference|Inv|Invoice)[:\s]*([A-Z0-9-]+)',
    }
    
    lines = text.split('\n')
    
    for line_num, line in enumerate(lines):
        line = line.strip()
        if not line or len(line) < 10:  # Skip very short lines
            continue
        
        # Check if line contains financial data
        has_date = re.search(patterns['date'], line, re.IGNORECASE)
        has_amount = re.search(patterns['amount'], line)
        
        if has_date or has_amount:
            transaction = {
                'line_number': line_num + 1,
                'raw_text': line[:200],  # First 200 chars
                'date': '',
                'description': '',
                'debit': '',
                'credit': '',
                'balance': '',
                'account_type': '',
                'reference': ''
            }
            
            # Extract date
            date_match = re.search(patterns['date'], line, re.IGNORECASE)
            if date_match:
                transaction['date'] = date_match.group(0)
            
            # Extract amounts
            amounts = re.findall(patterns['amount'], line)
            
            # Try to categorize amounts
            if 'debit' in line.lower() and amounts:
                transaction['debit'] = amounts[0]
            elif 'credit' in line.lower() and amounts:
                transaction['credit'] = amounts[0]
            elif 'balance' in line.lower() and amounts:
                transaction['balance'] = amounts[-1]
            elif len(amounts) >= 2:
                # Assume first is amount, last is balance
                transaction['debit'] = amounts[0]
                transaction['balance'] = amounts[-1]
            elif len(amounts) == 1:
                transaction['balance'] = amounts[0]
            
            # Extract reference
            ref_match = re.search(patterns['reference'], line, re.IGNORECASE)
            if ref_match:
                transaction['reference'] = ref_match.group(1)
            
            # Get description (text between date and first amount)
            if transaction['date']:
                parts = line.split(transaction['date'])
                if len(parts) > 1:
                    desc = parts[1].strip()
                    # Remove amounts from description
                    for amt in amounts:
                        desc = desc.replace(amt, '')
                    transaction['description'] = desc[:100]  # Limit length
            
            transactions.append(transaction)
    
    return transactions

def save_to_csv(transactions, output_path):
    """
    Save transactions to CSV file
    """
    if not transactions:
        print("No transactions to save")
        return False
    
    # Define CSV columns
    fieldnames = ['date', 'description', 'debit', 'credit', 'balance', 
                  'reference', 'account_type', 'line_number', 'raw_text']
    
    try:
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(transactions)
        
        print(f"✅ Successfully saved {len(transactions)} transactions to {output_path}")
        return True
    
    except Exception as e:
        print(f"❌ Error saving to CSV: {e}")
        return False

def create_human_readable_report(transactions, output_path):
    """
    Create a human-readable text report
    """
    if not transactions:
        return
    
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("FINANCIAL TRANSACTIONS EXTRACTED FROM PDF\n")
            f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("=" * 80 + "\n\n")
            
            for i, t in enumerate(transactions, 1):
                f.write(f"Transaction #{i}\n")
                f.write("-" * 40 + "\n")
                f.write(f"Date: {t.get('date', 'N/A')}\n")
                f.write(f"Description: {t.get('description', 'N/A')}\n")
                f.write(f"Debit: {t.get('debit', 'N/A')}\n")
                f.write(f"Credit: {t.get('credit', 'N/A')}\n")
                f.write(f"Balance: {t.get('balance', 'N/A')}\n")
                f.write(f"Reference: {t.get('reference', 'N/A')}\n")
                f.write(f"Raw: {t.get('raw_text', 'N/A')}\n")
                f.write("\n")
        
        print(f"✅ Human-readable report saved to {output_path}")
    
    except Exception as e:
        print(f"❌ Error creating report: {e}")

def main():
    """
    Main function
    """
    print("=" * 60)
    print("PDF FINANCIAL DATA EXTRACTOR")
    print("(No external libraries required)")
    print("=" * 60)
    
    # Your specific file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    print(f"\n📁 Processing file: {pdf_path}")
    
    # Check if input file exists
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        print(f"\n❌ Error: File not found!")
        print(f"   Please check the path: {pdf_path}")
        
        # Try to find the file in common locations
        print("\n🔍 Searching for the file...")
        common_paths = [
            r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES",
            r"C:\Users\Chigumira\Documents",
            r"C:\Users\Chigumira\Desktop"
        ]
        
        for base_path in common_paths:
            if os.path.exists(base_path):
                print(f"   Checking: {base_path}")
                for file in os.listdir(base_path):
                    if 'cash' in file.lower() and 'book' in file.lower():
                        found_path = os.path.join(base_path, file)
                        print(f"   ✓ Found possible file: {found_path}")
                        pdf_path = found_path
                        pdf_file = Path(pdf_path)
                        break
                if pdf_file.exists():
                    break
        
        if not pdf_file.exists():
            print("\n❌ Could not find the file automatically.")
            print("Please update the pdf_path variable in the script with the correct path.")
            return
    
    # Generate output filenames
    output_csv = pdf_file.parent / (pdf_file.stem + "_transactions.csv")
    output_txt = pdf_file.parent / (pdf_file.stem + "_report.txt")
    
    # Step 1: Extract text
    print("\n📖 Extracting text from PDF...")
    text_data = extract_text_from_pdf_simple(pdf_path)
    
    if not text_data:
        print("Trying alternative extraction method...")
        text_data = extract_text_alternative(pdf_path)
    
    if not text_data:
        print("❌ Could not extract any text from the PDF.")
        return
    
    print(f"\n✓ Extracted {len(text_data)} text blocks/pages")
    
    # Step 2: Parse transactions
    print("\n🔍 Parsing transactions...")
    all_transactions = []
    
    for block in text_data:
        transactions = parse_transactions(block['text'])
        if transactions:
            all_transactions.extend(transactions)
            print(f"  Found {len(transactions)} potential transactions in block {block['page']}")
    
    # Step 3: Save results
    if all_transactions:
        print(f"\n✓ Total potential transactions found: {len(all_transactions)}")
        
        # Save to CSV
        print("\n💾 Saving to CSV...")
        save_to_csv(all_transactions, output_csv)
        
        # Create human-readable report
        print("\n📝 Creating report...")
        create_human_readable_report(all_transactions, output_txt)
        
        # Show sample
        print("\n" + "=" * 60)
        print("SAMPLE EXTRACTED DATA (First 5 transactions)")
        print("=" * 60)
        for i, t in enumerate(all_transactions[:5]):
            print(f"\n{i+1}. Date: {t.get('date', 'N/A')}")
            print(f"   Desc: {t.get('description', 'N/A')[:50]}")
            print(f"   Debit: {t.get('debit', 'N/A')} | Credit: {t.get('credit', 'N/A')} | Balance: {t.get('balance', 'N/A')}")
        
        print(f"\n✅ Extraction complete!")
        print(f"   CSV file: {output_csv}")
        print(f"   Report file: {output_txt}")
        
    else:
        print("❌ No transactions could be parsed from the text.")
        
        # Show raw text sample for debugging
        print("\n📄 Raw text sample (first 1000 characters):")
        if text_data:
            print(text_data[0]['text'][:1000])

if __name__ == "__main__":
    main()

PDF FINANCIAL DATA EXTRACTOR
(No external libraries required)

📁 Processing file: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf

📖 Extracting text from PDF...
✓ Extracted text from page 1

✓ Extracted 1 text blocks/pages

🔍 Parsing transactions...
  Found 1 potential transactions in block 1

✓ Total potential transactions found: 1

💾 Saving to CSV...
✅ Successfully saved 1 transactions to C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_transactions.csv

📝 Creating report...
✅ Human-readable report saved to C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_report.txt

SAMPLE EXTRACTED DATA (First 5 transactions)

1. Date: 
   Desc: 
   Debit:  | Credit:  | Balance: ,.72

✅ Extraction complete!
   CSV file: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_transactions.csv
   Report fi

In [ ]:
import re
import os
from pathlib import Path
import csv
from datetime import datetime

def read_pdf_as_text(pdf_path):
    """
    Read PDF and extract text content
    """
    text_content = []
    
    try:
        with open(pdf_path, 'rb') as file:
            content = file.read().decode('latin-1', errors='ignore')
        
        # Find text blocks between BT and ET operators
        text_blocks = re.findall(r'BT(.*?)ET', content, re.DOTALL)
        
        for block in text_blocks:
            # Extract text between parentheses
            text_parts = re.findall(r'\((.*?)\)', block)
            if text_parts:
                # Clean up the text
                clean_text = ' '.join(text_parts)
                # Fix common PDF encoding
                clean_text = clean_text.replace('\\', '')
                # Remove extra spaces
                clean_text = re.sub(r'\s+', ' ', clean_text)
                text_content.append(clean_text)
        
        return ' '.join(text_content)
    
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ""

def parse_cashbook_format(text):
    """
    Parse the specific cash book format with columns:
    DATE, SOURCE, DESCRIPTION, REFERENCE, CURRENCY, 
    DEBIT(SOURCE), CREDIT(SOURCE), DEBIT(USD), CREDIT(USD),
    RUNNING BALANCE(USD), RELATED ACCOUNT
    """
    transactions = []
    
    # Split text into lines
    lines = text.split('\n')
    
    # Patterns for different data types
    date_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'
    amount_pattern = r'([\d,]+\.\d{2})'
    reference_pattern = r'[A-Z0-9]{5,}'
    
    current_transaction = {}
    line_buffer = []
    
    for line_num, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
            
        # Check if line starts with a date (new transaction)
        date_match = re.search(date_pattern, line)
        
        if date_match and len(line) > 10:  # Likely a new transaction
            # Save previous transaction if exists
            if current_transaction and any(current_transaction.values()):
                transactions.append(current_transaction)
            
            # Start new transaction
            current_transaction = {
                'DATE': '',
                'SOURCE': '',
                'DESCRIPTION': '',
                'REFERENCE': '',
                'CURRENCY': '',
                'DEBIT(SOURCE)': '',
                'CREDIT(SOURCE)': '',
                'DEBIT(USD)': '',
                'CREDIT(USD)': '',
                'RUNNING BALANCE(USD)': '',
                'RELATED ACCOUNT': '',
                'RAW_LINE': line
            }
            
            # Parse the line
            current_transaction['DATE'] = date_match.group(0)
            
            # Extract amounts
            amounts = re.findall(amount_pattern, line)
            
            # Try to identify currency
            if 'USD' in line:
                current_transaction['CURRENCY'] = 'USD'
            elif 'ZWL' in line:
                current_transaction['CURRENCY'] = 'ZWL'
            
            # Look for reference (alphanumeric code)
            ref_match = re.search(r'\b([A-Z0-9]{5,})\b', line)
            if ref_match:
                current_transaction['REFERENCE'] = ref_match.group(1)
            
            # The rest of the line might contain source and description
            # Remove date and amounts to get description
            desc_part = line
            if current_transaction['DATE']:
                desc_part = desc_part.replace(current_transaction['DATE'], '')
            for amt in amounts:
                desc_part = desc_part.replace(amt, '')
            if current_transaction['REFERENCE']:
                desc_part = desc_part.replace(current_transaction['REFERENCE'], '')
            
            # Clean up description
            desc_part = re.sub(r'\s+', ' ', desc_part).strip()
            
            # Try to split into SOURCE and DESCRIPTION
            # Look for common source indicators
            source_match = re.search(r'(Cash|Bank|Transfer|Payment|Receipt|Salary|Fee|Dividend|Interest)', desc_part, re.IGNORECASE)
            if source_match:
                source_end = source_match.end()
                current_transaction['SOURCE'] = source_match.group(0)
                current_transaction['DESCRIPTION'] = desc_part[source_end:].strip()
            else:
                current_transaction['DESCRIPTION'] = desc_part
            
            # Assign amounts based on position
            if len(amounts) >= 1:
                current_transaction['DEBIT(SOURCE)'] = amounts[0] if 'debit' in line.lower() else ''
                current_transaction['CREDIT(SOURCE)'] = amounts[0] if 'credit' in line.lower() else ''
            if len(amounts) >= 2:
                current_transaction['DEBIT(USD)'] = amounts[1] if 'debit' in line.lower() else ''
                current_transaction['CREDIT(USD)'] = amounts[1] if 'credit' in line.lower() else ''
            if len(amounts) >= 3:
                current_transaction['RUNNING BALANCE(USD)'] = amounts[-1]
        
        else:
            # This might be a continuation of the previous transaction
            if current_transaction:
                # Append to description
                current_transaction['DESCRIPTION'] += ' ' + line
                
                # Look for additional amounts in continuation lines
                amounts = re.findall(amount_pattern, line)
                if amounts and not current_transaction['RUNNING BALANCE(USD)']:
                    current_transaction['RUNNING BALANCE(USD)'] = amounts[-1]
    
    # Add the last transaction
    if current_transaction and any(current_transaction.values()):
        transactions.append(current_transaction)
    
    return transactions

def extract_account_sections(text):
    """
    Extract different account sections from the cash book
    """
    sections = []
    current_section = {'name': 'GENERAL', 'transactions': []}
    
    # Look for account headers
    account_patterns = [
        r'(Compensation.*?Account)',
        r'(Insurance.*?Proceeds.*?Account)',
        r'(.*?Zim.*?Account)',
        r'(Cash.*?Book)',
        r'(Bank.*?Statement)'
    ]
    
    lines = text.split('\n')
    
    for line in lines:
        line = line.strip()
        
        # Check if this line is an account header
        is_header = False
        for pattern in account_patterns:
            if re.search(pattern, line, re.IGNORECASE):
                # Save previous section
                if current_section['transactions']:
                    sections.append(current_section.copy())
                
                # Start new section
                current_section = {
                    'name': line.strip(),
                    'transactions': []
                }
                is_header = True
                break
        
        # If not a header and line contains transaction-like data, add to current section
        if not is_header and re.search(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', line):
            current_section['transactions'].append(line)
    
    # Add the last section
    if current_section['transactions']:
        sections.append(current_section)
    
    return sections

def save_to_csv(transactions, output_path):
    """
    Save transactions to CSV file with proper headers
    """
    if not transactions:
        print("No transactions to save")
        return False
    
    # Define CSV columns based on your format
    fieldnames = ['DATE', 'SOURCE', 'DESCRIPTION', 'REFERENCE', 'CURRENCY',
                  'DEBIT(SOURCE)', 'CREDIT(SOURCE)', 'DEBIT(USD)', 'CREDIT(USD)',
                  'RUNNING BALANCE(USD)', 'RELATED ACCOUNT', 'RAW_LINE']
    
    try:
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            
            for transaction in transactions:
                # Ensure all fields exist
                row = {field: transaction.get(field, '') for field in fieldnames}
                writer.writerow(row)
        
        print(f"✅ Successfully saved {len(transactions)} transactions to {output_path}")
        
        # Also save a summary
        summary_path = output_path.replace('.csv', '_summary.txt')
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("CASH BOOK EXTRACTION SUMMARY\n")
            f.write("=" * 80 + "\n")
            f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total transactions: {len(transactions)}\n\n")
            
            # Group by account
            accounts = {}
            for t in transactions:
                account = t.get('RELATED ACCOUNT', 'UNKNOWN')
                if account not in accounts:
                    accounts[account] = []
                accounts[account].append(t)
            
            for account, trans in accounts.items():
                f.write(f"\n{account}: {len(trans)} transactions\n")
                f.write("-" * 40 + "\n")
                
                # Calculate totals
                total_debit = 0
                total_credit = 0
                for t in trans:
                    if t.get('DEBIT(USD)'):
                        try:
                            total_debit += float(t['DEBIT(USD)'].replace(',', ''))
                        except:
                            pass
                    if t.get('CREDIT(USD)'):
                        try:
                            total_credit += float(t['CREDIT(USD)'].replace(',', ''))
                        except:
                            pass
                
                f.write(f"Total Debit (USD): {total_debit:,.2f}\n")
                f.write(f"Total Credit (USD): {total_credit:,.2f}\n")
        
        return True
    
    except Exception as e:
        print(f"❌ Error saving to CSV: {e}")
        return False

def main():
    """
    Main function
    """
    print("=" * 70)
    print("CASH BOOK PDF EXTRACTOR")
    print("For format: DATE | SOURCE | DESCRIPTION | REFERENCE | CURRENCY | DEBIT/CREDIT | BALANCE")
    print("=" * 70)
    
    # Your specific file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    print(f"\n📁 Processing file: {pdf_path}")
    
    # Check if input file exists
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        print(f"\n❌ Error: File not found!")
        print(f"   Please check the path: {pdf_path}")
        
        # Try to find the file
        if os.path.exists(r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES"):
            print("\n✓ Directory exists. Available files:")
            for file in os.listdir(r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES"):
                if file.endswith('.pdf'):
                    print(f"   - {file}")
        return
    
    # Generate output filename
    output_csv = pdf_file.parent / (pdf_file.stem + "_extracted.csv")
    
    # Step 1: Extract text
    print("\n📖 Extracting text from PDF...")
    text = read_pdf_as_text(pdf_path)
    
    if not text:
        print("❌ Could not extract text from the PDF.")
        return
    
    print(f"✓ Extracted {len(text)} characters of text")
    
    # Step 2: Identify account sections
    print("\n📑 Identifying account sections...")
    sections = extract_account_sections(text)
    
    if sections:
        print(f"✓ Found {len(sections)} account sections:")
        for section in sections:
            print(f"   - {section['name']}: {len(section['transactions'])} transactions")
    
    # Step 3: Parse transactions
    print("\n🔍 Parsing transactions in cash book format...")
    transactions = parse_cashbook_format(text)
    
    # Step 4: Save results
    if transactions:
        print(f"\n✓ Successfully parsed {len(transactions)} transactions")
        
        # Save to CSV
        print("\n💾 Saving to CSV...")
        save_to_csv(transactions, output_csv)
        
        # Show sample
        print("\n" + "=" * 70)
        print("SAMPLE EXTRACTED DATA (First 10 transactions)")
        print("=" * 70)
        
        # Create a formatted table for display
        print(f"{'DATE':<12} {'SOURCE':<15} {'DESCRIPTION':<30} {'REF':<10} {'CUR':<5} {'DEBIT':<12} {'CREDIT':<12} {'BALANCE':<12}")
        print("-" * 110)
        
        for i, t in enumerate(transactions[:10]):
            date = t.get('DATE', '')[:12]
            source = t.get('SOURCE', '')[:15]
            desc = t.get('DESCRIPTION', '')[:30]
            ref = t.get('REFERENCE', '')[:10]
            cur = t.get('CURRENCY', '')[:5]
            debit = t.get('DEBIT(SOURCE)', '')[:12]
            credit = t.get('CREDIT(SOURCE)', '')[:12]
            balance = t.get('RUNNING BALANCE(USD)', '')[:12]
            
            print(f"{date:<12} {source:<15} {desc:<30} {ref:<10} {cur:<5} {debit:<12} {credit:<12} {balance:<12}")
        
        print(f"\n✅ Extraction complete!")
        print(f"   Output file: {output_csv}")
        
    else:
        print("❌ No transactions could be parsed.")
        print("\n📄 Raw text preview (first 500 characters):")
        print(text[:500])

if __name__ == "__main__":
    main()

CASH BOOK PDF EXTRACTOR
For format: DATE | SOURCE | DESCRIPTION | REFERENCE | CURRENCY | DEBIT/CREDIT | BALANCE

📁 Processing file: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf

📖 Extracting text from PDF...
✓ Extracted 32527054 characters of text

📑 Identifying account sections...


In [1]:
import re
import os
from pathlib import Path
import csv

def quick_extract(pdf_path):
    """
    Quick extraction of cash book data - should complete in seconds
    """
    print("=" * 60)
    print("QUICK CASH BOOK EXTRACTOR")
    print("=" * 60)
    
    print(f"\n📁 Processing: {pdf_path}")
    
    try:
        # Read the file in binary mode and decode quickly
        with open(pdf_path, 'rb') as file:
            # Only read first 10MB to keep it fast
            content = file.read(10_000_000).decode('latin-1', errors='ignore')
        
        print(f"✓ Read {len(content):,} characters")
        
        # Look for patterns that look like transaction lines
        transactions = []
        
        # Split into lines
        lines = content.split('\n')
        print(f"✓ Processing {len(lines)} lines")
        
        # Common patterns in your cash book
        date_pattern = r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}'
        amount_pattern = r'[\d,]+\.\d{2}'
        
        for i, line in enumerate(lines):
            line = line.strip()
            if len(line) < 20:  # Skip short lines
                continue
            
            # Look for lines that contain both a date and an amount
            has_date = re.search(date_pattern, line)
            has_amount = re.search(amount_pattern, line)
            
            if has_date and has_amount:
                # Extract data
                date_match = re.search(date_pattern, line)
                amounts = re.findall(amount_pattern, line)
                
                # Clean the line - remove PDF artifacts
                clean_line = re.sub(r'[^\x20-\x7E]', ' ', line)  # Remove non-printable chars
                clean_line = re.sub(r'\s+', ' ', clean_line)  # Normalize spaces
                
                # Try to determine currency
                currency = 'USD' if 'USD' in clean_line.upper() else 'ZWL' if 'ZWL' in clean_line.upper() else ''
                
                # Create transaction record
                transaction = {
                    'LINE': i + 1,
                    'DATE': date_match.group(0) if date_match else '',
                    'DESCRIPTION': clean_line[:100],
                    'AMOUNTS': ' | '.join(amounts),
                    'CURRENCY': currency,
                    'RAW_TEXT': clean_line[:200]
                }
                
                # Try to identify if it's a debit or credit
                if 'DEBIT' in clean_line.upper():
                    transaction['TYPE'] = 'DEBIT'
                elif 'CREDIT' in clean_line.upper():
                    transaction['TYPE'] = 'CREDIT'
                else:
                    transaction['TYPE'] = 'TRANSACTION'
                
                transactions.append(transaction)
                
                # Stop after 1000 transactions to keep it fast
                if len(transactions) >= 1000:
                    break
        
        print(f"✓ Found {len(transactions)} potential transactions")
        return transactions
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return []

def save_quick_results(transactions, output_path):
    """
    Save results to CSV quickly
    """
    if not transactions:
        print("No transactions to save")
        return
    
    try:
        with open(output_path, 'w', newline='', encoding='utf-8') as f:
            # Write header
            f.write("LINE_NUMBER,DATE,DESCRIPTION,AMOUNTS,CURRENCY,TYPE,RAW_TEXT\n")
            
            # Write data
            for t in transactions:
                # Escape quotes in text fields
                desc = t.get('DESCRIPTION', '').replace('"', '""')
                raw = t.get('RAW_TEXT', '').replace('"', '""')
                
                f.write(f"{t.get('LINE', '')},\"{t.get('DATE', '')}\",\"{desc}\",\"{t.get('AMOUNTS', '')}\",{t.get('CURRENCY', '')},{t.get('TYPE', '')},\"{raw}\"\n")
        
        print(f"\n✅ Saved {len(transactions)} transactions to {output_path}")
        
        # Also save a simple text file with raw data
        txt_path = output_path.replace('.csv', '_raw.txt')
        with open(txt_path, 'w', encoding='utf-8') as f:
            for t in transactions[:50]:  # First 50 only
                f.write(f"Line {t['LINE']}: {t['RAW_TEXT']}\n")
        
        print(f"✅ Raw data preview saved to {txt_path}")
        
    except Exception as e:
        print(f"❌ Error saving: {e}")

def main():
    # Your file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return
    
    # Extract data (should take seconds, not minutes)
    transactions = quick_extract(pdf_path)
    
    if transactions:
        # Save results
        output_csv = Path(pdf_path).parent / (Path(pdf_path).stem + "_quick_extract.csv")
        save_quick_results(transactions, output_csv)
        
        # Show sample
        print("\n" + "=" * 60)
        print("SAMPLE EXTRACTED DATA (First 10 lines)")
        print("=" * 60)
        for i, t in enumerate(transactions[:10]):
            print(f"\n{i+1}. Line {t['LINE']}")
            print(f"   Date: {t['DATE']}")
            print(f"   Description: {t['DESCRIPTION'][:50]}...")
            print(f"   Amounts: {t['AMOUNTS']}")
            print(f"   Currency: {t['CURRENCY']}")
    else:
        print("❌ No transactions found")

if __name__ == "__main__":
    main()

QUICK CASH BOOK EXTRACTOR

📁 Processing: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf
✓ Read 10,000,000 characters
✓ Processing 32105 lines
✓ Found 0 potential transactions
❌ No transactions found


In [2]:
import re
import os
from pathlib import Path

def extract_binary_patterns(pdf_path):
    """
    Extract patterns from binary PDF data
    """
    print("=" * 60)
    print("BINARY PATTERN EXTRACTOR")
    print("=" * 60)
    
    print(f"\n📁 Processing: {pdf_path}")
    
    try:
        # Read the file in binary mode
        with open(pdf_path, 'rb') as file:
            raw_data = file.read()
        
        print(f"✓ Read {len(raw_data):,} bytes")
        
        # Try different encodings
        encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
        
        for encoding in encodings:
            try:
                text = raw_data.decode(encoding, errors='ignore')
                
                # Look for patterns that might be dates
                date_patterns = [
                    r'20\d{2}[/-]\d{1,2}[/-]\d{1,2}',  # 2023/12/31
                    r'\d{1,2}[/-]\d{1,2}[/-]20\d{2}',  # 31/12/2023
                    r'\d{1,2}[/-]\d{1,2}[/-]\d{2}',    # 31/12/23
                ]
                
                # Look for amount patterns
                amount_patterns = [
                    r'\d+\.\d{2}',           # 1234.56
                    r'\d{1,3}(?:,\d{3})*\.\d{2}',  # 1,234.56
                ]
                
                found_dates = []
                found_amounts = []
                
                for pattern in date_patterns:
                    found_dates.extend(re.findall(pattern, text))
                
                for pattern in amount_patterns:
                    found_amounts.extend(re.findall(pattern, text))
                
                if found_dates or found_amounts:
                    print(f"\n✓ Found data with {encoding} encoding:")
                    print(f"  Dates found: {len(found_dates)}")
                    print(f"  Amounts found: {len(found_amounts)}")
                    
                    # Show some samples
                    if found_dates:
                        print("\n📅 Sample dates (first 10):")
                        for d in found_dates[:10]:
                            print(f"  - {d}")
                    
                    if found_amounts:
                        print("\n💰 Sample amounts (first 10):")
                        for a in found_amounts[:10]:
                            print(f"  - {a}")
                    
                    return found_dates, found_amounts, encoding
                    
            except Exception as e:
                continue
        
        print("❌ No patterns found with any encoding")
        return [], [], None
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return [], [], None

def extract_text_blocks(pdf_path):
    """
    Extract text blocks from PDF
    """
    try:
        with open(pdf_path, 'rb') as file:
            data = file.read()
        
        # Convert to string
        text = data.decode('latin-1', errors='ignore')
        
        # Find all text between BT and ET operators (PDF text blocks)
        text_blocks = re.findall(r'BT(.*?)ET', text, re.DOTALL)
        
        print(f"\n📄 Found {len(text_blocks)} text blocks")
        
        all_text = []
        for i, block in enumerate(text_blocks[:50]):  # First 50 blocks
            # Extract text between parentheses
            text_parts = re.findall(r'\((.*?)\)', block)
            if text_parts:
                block_text = ' '.join(text_parts)
                # Clean up
                block_text = re.sub(r'[^\x20-\x7E]', ' ', block_text)
                block_text = re.sub(r'\s+', ' ', block_text).strip()
                
                if len(block_text) > 10:  # Only substantial blocks
                    all_text.append(block_text)
                    print(f"\nBlock {i+1}: {block_text[:100]}...")
        
        return all_text
        
    except Exception as e:
        print(f"Error extracting text blocks: {e}")
        return []

def main():
    # Your file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return
    
    # Try binary pattern extraction first
    dates, amounts, encoding = extract_binary_patterns(pdf_path)
    
    if not dates and not amounts:
        print("\n Trying text block extraction...")
        text_blocks = extract_text_blocks(pdf_path)
        
        if text_blocks:
            print(f"\n✅ Found {len(text_blocks)} text blocks")
            
            # Save to file
            output_txt = Path(pdf_path).parent / (Path(pdf_path).stem + "_extracted_blocks.txt")
            with open(output_txt, 'w', encoding='utf-8') as f:
                for i, block in enumerate(text_blocks):
                    f.write(f"BLOCK {i+1}:\n{block}\n{'-'*50}\n")
            
            print(f"\n✅ Text blocks saved to: {output_txt}")
            
            # Show first few blocks
            print("\n📄 First 5 text blocks:")
            for i, block in enumerate(text_blocks[:5]):
                print(f"\nBlock {i+1}:")
                print(block[:200] + "..." if len(block) > 200 else block)

if __name__ == "__main__":
    main()

BINARY PATTERN EXTRACTOR

📁 Processing: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf
✓ Read 344,793,960 bytes

✓ Found data with utf-8 encoding:
  Dates found: 17
  Amounts found: 850

📅 Sample dates (first 10):
  - 2023/06/11
  - 2026-02-09
  - 2026-02-09
  - 2023/06/11
  - 2026-02-09
  - 2026-02-09
  - 2026-02-09
  - 23/06/11
  - 99/02/22
  - 26-02-09

💰 Sample amounts (first 10):
  - 4.16
  - 9.51
  - 8.4۸
  - 9.93
  - 3.90
  - 8.71
  - 4.76
  - 9.78
  - 5.68
  - 1.95


In [3]:
import re
import os
from pathlib import Path
import csv
from datetime import datetime

def extract_cashbook_data(pdf_path):
    """
    Extract cash book data from PDF with proper handling of special characters
    """
    print("=" * 70)
    print("CASH BOOK DATA EXTRACTOR")
    print("=" * 70)
    
    print(f"\n📁 Processing: {pdf_path}")
    
    try:
        # Read the file
        with open(pdf_path, 'rb') as file:
            raw_data = file.read()
        
        # Decode with utf-8
        text = raw_data.decode('utf-8', errors='ignore')
        
        print(f"✓ Read {len(raw_data):,} bytes")
        print(f"✓ Decoded to {len(text):,} characters")
        
        # Clean the text - replace problematic characters
        # Convert Persian/Arabic digits to English
        persian_digits = {
            '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
            '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9'
        }
        
        for persian, english in persian_digits.items():
            text = text.replace(persian, english)
        
        # Also replace other common problematic characters
        text = re.sub(r'[^\x20-\x7E\n\r\t]', ' ', text)  # Keep only printable ASCII plus newlines/tabs
        text = re.sub(r'\s+', ' ', text)  # Normalize spaces
        
        # Split into lines
        lines = text.split('\n')
        print(f"✓ Processing {len(lines)} lines")
        
        # Patterns for data extraction
        date_patterns = [
            r'20\d{2}[/-]\d{1,2}[/-]\d{1,2}',  # 2023/06/11
            r'\d{1,2}[/-]\d{1,2}[/-]20\d{2}',  # 11/06/2023
            r'\d{1,2}[/-]\d{1,2}[/-]\d{2}',    # 11/06/23
        ]
        
        amount_pattern = r'(\d+\.\d{2})'  # Simple decimal pattern
        
        transactions = []
        current_transaction = {}
        
        for line_num, line in enumerate(lines):
            line = line.strip()
            if len(line) < 10:  # Skip very short lines
                continue
            
            # Look for dates
            date_found = None
            for pattern in date_patterns:
                date_match = re.search(pattern, line)
                if date_match:
                    date_found = date_match.group(0)
                    break
            
            # Look for amounts
            amounts = re.findall(amount_pattern, line)
            
            # If we found either a date or amounts, this might be a transaction
            if date_found or amounts:
                # Clean the line
                clean_line = re.sub(r'[^\x20-\x7E]', ' ', line)
                clean_line = re.sub(r'\s+', ' ', clean_line).strip()
                
                transaction = {
                    'LINE_NUM': line_num + 1,
                    'DATE': date_found if date_found else '',
                    'RAW_TEXT': clean_line[:200],
                    'AMOUNTS': '|'.join(amounts) if amounts else '',
                    'AMOUNT_COUNT': len(amounts),
                }
                
                # Try to identify transaction type
                if 'DEBIT' in clean_line.upper() or 'DR' in clean_line.upper():
                    transaction['TYPE'] = 'DEBIT'
                elif 'CREDIT' in clean_line.upper() or 'CR' in clean_line.upper():
                    transaction['TYPE'] = 'CREDIT'
                elif 'BALANCE' in clean_line.upper() or 'BAL' in clean_line.upper():
                    transaction['TYPE'] = 'BALANCE'
                else:
                    transaction['TYPE'] = 'TRANSACTION'
                
                # Try to identify currency
                if 'USD' in clean_line.upper():
                    transaction['CURRENCY'] = 'USD'
                elif 'ZWL' in clean_line.upper() or 'ZIM' in clean_line.upper():
                    transaction['CURRENCY'] = 'ZWL'
                else:
                    transaction['CURRENCY'] = 'UNKNOWN'
                
                transactions.append(transaction)
        
        print(f"✓ Found {len(transactions)} potential transaction lines")
        return transactions
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return []

def save_transactions(transactions, output_path):
    """
    Save transactions to CSV
    """
    if not transactions:
        print("No transactions to save")
        return False
    
    try:
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as csvfile:
            fieldnames = ['LINE_NUM', 'DATE', 'TYPE', 'CURRENCY', 'AMOUNTS', 'AMOUNT_COUNT', 'RAW_TEXT']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            writer.writeheader()
            for t in transactions:
                # Clean any remaining problematic characters
                clean_t = {k: (str(v).replace('\x00', '') if v else '') for k, v in t.items()}
                writer.writerow(clean_t)
        
        print(f"\n✅ Saved {len(transactions)} transactions to {output_path}")
        
        # Create a summary file
        summary_path = output_path.replace('.csv', '_summary.txt')
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("CASH BOOK EXTRACTION SUMMARY\n")
            f.write("=" * 60 + "\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total transaction lines: {len(transactions)}\n\n")
            
            # Count by currency
            usd_count = sum(1 for t in transactions if t.get('CURRENCY') == 'USD')
            zwl_count = sum(1 for t in transactions if t.get('CURRENCY') == 'ZWL')
            f.write(f"USD transactions: {usd_count}\n")
            f.write(f"ZWL transactions: {zwl_count}\n\n")
            
            # Count by type
            debit_count = sum(1 for t in transactions if t.get('TYPE') == 'DEBIT')
            credit_count = sum(1 for t in transactions if t.get('TYPE') == 'CREDIT')
            f.write(f"Debit transactions: {debit_count}\n")
            f.write(f"Credit transactions: {credit_count}\n\n")
            
            # Sample transactions
            f.write("SAMPLE TRANSACTIONS (First 20):\n")
            f.write("-" * 60 + "\n")
            for i, t in enumerate(transactions[:20]):
                f.write(f"{i+1}. Line {t['LINE_NUM']} | {t['DATE']} | {t['TYPE']} | {t['CURRENCY']} | Amounts: {t['AMOUNTS']}\n")
                f.write(f"   Text: {t['RAW_TEXT'][:100]}...\n\n")
        
        print(f"✅ Summary saved to {summary_path}")
        return True
        
    except Exception as e:
        print(f"❌ Error saving: {e}")
        return False

def extract_structured_data(transactions):
    """
    Try to extract structured data (if amounts are in specific positions)
    """
    print("\n Attempting structured data extraction...")
    
    structured = []
    
    for t in transactions:
        if t['AMOUNT_COUNT'] >= 3:  # Has multiple amounts (debit, credit, balance)
            amounts = t['AMOUNTS'].split('|')
            
            structured_transaction = {
                'DATE': t['DATE'],
                'TYPE': t['TYPE'],
                'CURRENCY': t['CURRENCY'],
                'DEBIT': amounts[0] if len(amounts) > 0 else '',
                'CREDIT': amounts[1] if len(amounts) > 1 else '',
                'BALANCE': amounts[-1] if len(amounts) > 0 else '',
                'RAW_TEXT': t['RAW_TEXT']
            }
            structured.append(structured_transaction)
    
    if structured:
        # Save structured data
        output_path = Path(pdf_path).parent / (Path(pdf_path).stem + "_structured.csv")
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as csvfile:
            fieldnames = ['DATE', 'TYPE', 'CURRENCY', 'DEBIT', 'CREDIT', 'BALANCE', 'RAW_TEXT']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(structured)
        
        print(f"✅ Saved {len(structured)} structured transactions to {output_path}")
    
    return structured

def main():
    global pdf_path  # Make pdf_path available to other functions
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return
    
    # Extract data
    transactions = extract_cashbook_data(pdf_path)
    
    if transactions:
        # Save to CSV
        output_csv = Path(pdf_path).parent / (Path(pdf_path).stem + "_extracted.csv")
        save_transactions(transactions, output_csv)
        
        # Try structured extraction
        structured = extract_structured_data(transactions)
        
        # Show sample
        print("\n" + "=" * 70)
        print("SAMPLE EXTRACTED DATA (First 20 lines)")
        print("=" * 70)
        
        for i, t in enumerate(transactions[:20]):
            print(f"\n{i+1}. Line {t['LINE_NUM']}")
            print(f"   Date: {t['DATE']}")
            print(f"   Type: {t['TYPE']} | Currency: {t['CURRENCY']}")
            print(f"   Amounts: {t['AMOUNTS']}")
            print(f"   Text: {t['RAW_TEXT'][:100]}")
        
        print(f"\n✅ Extraction complete!")
        print(f"   Output file: {output_csv}")
        
    else:
        print("❌ No transactions extracted")

if __name__ == "__main__":
    main()

CASH BOOK DATA EXTRACTOR

📁 Processing: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf
✓ Read 344,793,960 bytes
✓ Decoded to 180,583,366 characters
✓ Processing 1 lines
✓ Found 1 potential transaction lines

✅ Saved 1 transactions to C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_extracted.csv
❌ Error saving: Path.replace() takes 2 positional arguments but 3 were given

 Attempting structured data extraction...
✅ Saved 1 structured transactions to C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_structured.csv

SAMPLE EXTRACTED DATA (First 20 lines)

1. Line 1
   Date: 2023/06/11
   Type: DEBIT | Currency: USD
   Amounts: 4.16|9.51|8.48|9.93|3.90|8.71|4.76|9.78|5.68|1.95|9.07|4.47|5.56|9.43|2.33|8.15|9.51|9.78|9.78|9.43|8.48|6.94|5.64|1.03|7.42|2.79|9.78|11.41|8.43|1.47|9.03|2.11|3.75|9.55|3.75|7.93|9.78|2.76|3.85|2.76|0.76|3.90|4

In [5]:
import re
import os
from pathlib import Path
import csv

def clean_persian_digits(text):
    """Convert Persian/Arabic digits to English."""
    persian_digits = {
        '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
        '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9'
    }
    for p, e in persian_digits.items():
        text = text.replace(p, e)
    return text

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using binary read and utf-8 decoding."""
    try:
        with open(pdf_path, 'rb') as f:
            raw = f.read()
        # Decode with utf-8, ignoring errors
        text = raw.decode('utf-8', errors='ignore')
        # Clean Persian digits
        text = clean_persian_digits(text)
        # Replace other non-printable chars with space
        text = re.sub(r'[^\x20-\x7E\n\r\t]', ' ', text)
        # Normalize spaces
        text = re.sub(r' +', ' ', text)
        return text
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ""

def parse_cashbook_sections(text):
    """
    Split text into sections based on 'Opening Balance' and 'Closing Balance'.
    Each section has a heading line just before 'Opening Balance'.
    Returns list of dicts: {'heading': str, 'lines': list of str}
    """
    lines = text.split('\n')
    sections = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if 'Opening Balance' in line:
            # Look backwards for heading (skip empty lines)
            heading = "Unknown Account"
            j = i - 1
            while j >= 0 and not lines[j].strip():
                j -= 1
            if j >= 0:
                heading = lines[j].strip()
            # Collect lines until 'Closing Balance'
            section_lines = []
            k = i
            while k < len(lines) and 'Closing Balance' not in lines[k]:
                if lines[k].strip():
                    section_lines.append(lines[k].strip())
                k += 1
            # Include the closing balance line if found
            if k < len(lines) and 'Closing Balance' in lines[k]:
                section_lines.append(lines[k].strip())
            sections.append({
                'heading': heading,
                'lines': section_lines
            })
            i = k + 1
        else:
            i += 1
    return sections

def extract_transactions_from_section(section):
    """
    From a section's lines, extract lines that contain a date.
    For each such line, parse amounts into the expected columns.
    """
    # Date patterns including "01 Jan 2023"
    date_patterns = [
        r'\b\d{1,2} \w{3,9} \d{4}\b',          # 01 Jan 2023, 01 January 2023
        r'\b20\d{2}[/-]\d{1,2}[/-]\d{1,2}\b',   # 2023/06/11
        r'\b\d{1,2}[/-]\d{1,2}[/-]20\d{2}\b',   # 11/06/2023
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2}\b',     # 11/06/23
    ]
    amount_pattern = r'(\d+\.\d{2})'
    
    transactions = []
    for line in section['lines']:
        # Skip lines that are likely section markers
        if any(x in line for x in ['Opening Balance', 'Closing Balance']):
            continue
        
        # Find date
        date_found = None
        for pat in date_patterns:
            m = re.search(pat, line)
            if m:
                date_found = m.group(0)
                break
        
        if not date_found:
            continue
        
        # Extract all amounts
        amounts = re.findall(amount_pattern, line)
        amount_count = len(amounts)
        
        # Clean line for further analysis
        clean_line = re.sub(r'\s+', ' ', line).strip()
        
        # Determine currency: look for USD or ZWD (case-insensitive)
        currency = 'UNKNOWN'
        if re.search(r'\bUSD\b', line, re.IGNORECASE):
            currency = 'USD'
        elif re.search(r'\bZWD\b', line, re.IGNORECASE) or re.search(r'\bZWL\b', line, re.IGNORECASE):
            currency = 'ZWD'
        
        # Initialize structured fields
        debit_source = ''
        credit_source = ''
        debit_usd = ''
        credit_usd = ''
        balance_usd = ''
        
        # Attempt to map amounts based on count and position
        # If there are 4 or more amounts, assume first four are in order: DEBIT(SOURCE), CREDIT(SOURCE), DEBIT(USD), CREDIT(USD)
        # If 5 or more, the fifth is RUNNING BALANCE(USD)
        if amount_count >= 4:
            debit_source = amounts[0]
            credit_source = amounts[1]
            debit_usd = amounts[2]
            credit_usd = amounts[3]
            if amount_count >= 5:
                balance_usd = amounts[4]
        elif amount_count == 3:
            # Possibly SOURCE debit/credit and USD balance? Not sure, leave blank
            pass
        elif amount_count == 2:
            # Possibly SOURCE and USD amounts? Leave blank
            pass
        elif amount_count == 1:
            # Maybe just balance
            balance_usd = amounts[0]
        
        # Refine using keywords if present
        if re.search(r'DEBIT.*SOURCE', line, re.IGNORECASE) and not debit_source and amounts:
            debit_source = amounts[0] if amounts else ''
        if re.search(r'CREDIT.*SOURCE', line, re.IGNORECASE) and not credit_source and len(amounts) > 1:
            credit_source = amounts[1] if len(amounts) > 1 else ''
        if re.search(r'DEBIT.*USD', line, re.IGNORECASE) and not debit_usd and len(amounts) > 2:
            debit_usd = amounts[2] if len(amounts) > 2 else ''
        if re.search(r'CREDIT.*USD', line, re.IGNORECASE) and not credit_usd and len(amounts) > 3:
            credit_usd = amounts[3] if len(amounts) > 3 else ''
        if re.search(r'BALANCE', line, re.IGNORECASE) and not balance_usd and amounts:
            balance_usd = amounts[-1]  # last amount is often balance
        
        transactions.append({
            'ACCOUNT': section['heading'],
            'DATE': date_found,
            'CURRENCY': currency,
            'DEBIT(SOURCE)': debit_source,
            'CREDIT(SOURCE)': credit_source,
            'DEBIT(USD)': debit_usd,
            'CREDIT(USD)': credit_usd,
            'RUNNING BALANCE(USD)': balance_usd,
            'AMOUNTS_RAW': '|'.join(amounts),
            'RAW_TEXT': clean_line[:200]
        })
    
    return transactions

def save_transactions(transactions, output_path):
    """Save transactions to CSV."""
    if not transactions:
        print("No transactions to save.")
        return False
    
    fieldnames = [
        'ACCOUNT', 'DATE', 'CURRENCY',
        'DEBIT(SOURCE)', 'CREDIT(SOURCE)',
        'DEBIT(USD)', 'CREDIT(USD)',
        'RUNNING BALANCE(USD)',
        'AMOUNTS_RAW', 'RAW_TEXT'
    ]
    try:
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(transactions)
        print(f"✅ Saved {len(transactions)} transactions to {output_path}")
        return True
    except Exception as e:
        print(f"❌ Error saving CSV: {e}")
        return False

def main():
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    if not os.path.exists(pdf_path):
        print(f"File not found: {pdf_path}")
        return
    
    print("Extracting text from PDF...")
    text = extract_text_from_pdf(pdf_path)
    if not text:
        print("No text extracted.")
        return
    
    print("Parsing sections...")
    sections = parse_cashbook_sections(text)
    print(f"Found {len(sections)} table sections.")
    
    all_transactions = []
    for sec in sections:
        print(f"  Section: {sec['heading']} ({len(sec['lines'])} lines)")
        txns = extract_transactions_from_section(sec)
        all_transactions.extend(txns)
        print(f"    -> {len(txns)} transaction lines")
    
    if all_transactions:
        out_csv = Path(pdf_path).parent / (Path(pdf_path).stem + "_transactions.csv")
        save_transactions(all_transactions, out_csv)
        
        # Print sample
        print("\nSample transactions (first 10):")
        for i, t in enumerate(all_transactions[:10]):
            print(f"{i+1}. [{t['ACCOUNT'][:30]}] {t['DATE']} | {t['CURRENCY']} | "
                  f"DR src: {t['DEBIT(SOURCE)']} CR src: {t['CREDIT(SOURCE)']} | "
                  f"DR USD: {t['DEBIT(USD)']} CR USD: {t['CREDIT(USD)']} | Bal: {t['RUNNING BALANCE(USD)']}")
            print(f"   Raw: {t['RAW_TEXT'][:80]}...")
    else:
        print("No transactions found.")

if __name__ == "__main__":
    main()


Extracting text from PDF...
Parsing sections...
Found 0 table sections.
No transactions found.


In [6]:
import re
import os
from pathlib import Path
import csv

def clean_persian_digits(text):
    """Convert Persian/Arabic digits to English."""
    persian_digits = {
        '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
        '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9'
    }
    for p, e in persian_digits.items():
        text = text.replace(p, e)
    return text

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using binary read and utf-8 decoding."""
    try:
        with open(pdf_path, 'rb') as f:
            raw = f.read()
        # Decode with utf-8, ignoring errors
        text = raw.decode('utf-8', errors='ignore')
        # Clean Persian digits
        text = clean_persian_digits(text)
        # Replace other non-printable chars with space
        text = re.sub(r'[^\x20-\x7E\n\r\t]', ' ', text)
        # Normalize spaces
        text = re.sub(r' +', ' ', text)
        return text
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ""

def find_all_transactions(text):
    """
    Directly find all lines that contain dates without relying on sections.
    This is more robust if the PDF structure varies.
    """
    # Date patterns including "01 Jan 2023" and numeric formats
    date_patterns = [
        r'\b\d{1,2} \w{3,9} \d{4}\b',          # 01 Jan 2023, 01 January 2023
        r'\b20\d{2}[/-]\d{1,2}[/-]\d{1,2}\b',   # 2023/06/11
        r'\b\d{1,2}[/-]\d{1,2}[/-]20\d{2}\b',   # 11/06/2023
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2}\b',     # 11/06/23
    ]
    amount_pattern = r'(\d+\.\d{2})'
    
    # Split into lines
    lines = text.split('\n')
    transactions = []
    current_account = "Unknown Account"
    
    for line_num, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
        
        # Look for account headers (lines that might indicate a new section)
        # Common account indicators
        account_keywords = ['Compensation', 'Insurance', 'Proceeds', 'Account', 'Zim', 'ZWD', 'USD']
        if any(keyword in line for keyword in account_keywords) and len(line) < 100:
            # This might be an account header
            if 'balance' not in line.lower():  # Avoid capturing balance lines as headers
                current_account = line
                print(f"Found account header: {current_account}")
                continue
        
        # Check if line contains a date
        date_found = None
        for pat in date_patterns:
            m = re.search(pat, line)
            if m:
                date_found = m.group(0)
                break
        
        if not date_found:
            continue
        
        # Skip obvious header lines
        if any(x in line.lower() for x in ['opening balance', 'closing balance', 'account', 'date', 'description']):
            continue
        
        # Extract all amounts
        amounts = re.findall(amount_pattern, line)
        
        if not amounts:  # Skip if no amounts found
            continue
        
        # Clean line for further analysis
        clean_line = re.sub(r'\s+', ' ', line).strip()
        
        # Determine currency
        currency = 'UNKNOWN'
        if re.search(r'\bUSD\b', line, re.IGNORECASE):
            currency = 'USD'
        elif re.search(r'\bZWD\b', line, re.IGNORECASE) or re.search(r'\bZWL\b', line, re.IGNORECASE):
            currency = 'ZWD'
        
        # Initialize structured fields
        debit_source = ''
        credit_source = ''
        debit_usd = ''
        credit_usd = ''
        balance_usd = ''
        
        # Try to identify the type of transaction based on keywords
        if re.search(r'debit.*source|dr.*source', line, re.IGNORECASE):
            debit_source = amounts[0] if amounts else ''
        elif re.search(r'credit.*source|cr.*source', line, re.IGNORECASE):
            credit_source = amounts[0] if amounts else ''
        elif re.search(r'debit.*usd|dr.*usd', line, re.IGNORECASE):
            debit_usd = amounts[0] if amounts else ''
        elif re.search(r'credit.*usd|cr.*usd', line, re.IGNORECASE):
            credit_usd = amounts[0] if amounts else ''
        elif re.search(r'balance', line, re.IGNORECASE):
            balance_usd = amounts[-1] if amounts else ''
        else:
            # Generic transaction - try to assign based on amount count
            if len(amounts) >= 4:
                debit_source = amounts[0]
                credit_source = amounts[1]
                debit_usd = amounts[2]
                credit_usd = amounts[3]
                if len(amounts) >= 5:
                    balance_usd = amounts[4]
            elif len(amounts) == 3:
                # Could be source debit, source credit, balance
                debit_source = amounts[0]
                credit_source = amounts[1]
                balance_usd = amounts[2]
            elif len(amounts) == 2:
                # Could be debit/credit and balance
                # Check line for hints
                if 'debit' in line.lower() or 'dr' in line.lower():
                    debit_source = amounts[0]
                    balance_usd = amounts[1]
                elif 'credit' in line.lower() or 'cr' in line.lower():
                    credit_source = amounts[0]
                    balance_usd = amounts[1]
                else:
                    # Assume first is amount, second is balance
                    debit_source = amounts[0]
                    balance_usd = amounts[1]
            elif len(amounts) == 1:
                balance_usd = amounts[0]
        
        transactions.append({
            'ACCOUNT': current_account,
            'LINE_NUM': line_num + 1,
            'DATE': date_found,
            'CURRENCY': currency,
            'DEBIT(SOURCE)': debit_source,
            'CREDIT(SOURCE)': credit_source,
            'DEBIT(USD)': debit_usd,
            'CREDIT(USD)': credit_usd,
            'RUNNING BALANCE(USD)': balance_usd,
            'AMOUNTS_FOUND': len(amounts),
            'AMOUNTS_RAW': '|'.join(amounts),
            'RAW_TEXT': clean_line[:200]
        })
    
    return transactions

def save_transactions(transactions, output_path):
    """Save transactions to CSV."""
    if not transactions:
        print("No transactions to save.")
        return False
    
    fieldnames = [
        'ACCOUNT', 'LINE_NUM', 'DATE', 'CURRENCY',
        'DEBIT(SOURCE)', 'CREDIT(SOURCE)',
        'DEBIT(USD)', 'CREDIT(USD)',
        'RUNNING BALANCE(USD)',
        'AMOUNTS_FOUND', 'AMOUNTS_RAW', 'RAW_TEXT'
    ]
    try:
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(transactions)
        print(f"✅ Saved {len(transactions)} transactions to {output_path}")
        
        # Also save a debug file with all lines that have amounts
        debug_path = output_path.replace('.csv', '_debug.txt')
        with open(debug_path, 'w', encoding='utf-8') as f:
            f.write("LINES WITH AMOUNTS FOUND:\n")
            f.write("=" * 80 + "\n")
            for t in transactions:
                f.write(f"Line {t['LINE_NUM']}: {t['RAW_TEXT']}\n")
                f.write(f"Amounts: {t['AMOUNTS_RAW']}\n")
                f.write("-" * 40 + "\n")
        print(f"✅ Debug info saved to {debug_path}")
        
        return True
    except Exception as e:
        print(f"❌ Error saving CSV: {e}")
        return False

def main():
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    if not os.path.exists(pdf_path):
        print(f"File not found: {pdf_path}")
        return
    
    print("Extracting text from PDF...")
    text = extract_text_from_pdf(pdf_path)
    if not text:
        print("No text extracted.")
        return
    
    print(f"Extracted {len(text)} characters")
    print(f"First 500 characters of extracted text:")
    print("-" * 50)
    print(text[:500])
    print("-" * 50)
    
    print("\nSearching for transactions...")
    transactions = find_all_transactions(text)
    
    if transactions:
        print(f"\n✅ Found {len(transactions)} potential transaction lines")
        
        # Group by account
        accounts = {}
        for t in transactions:
            if t['ACCOUNT'] not in accounts:
                accounts[t['ACCOUNT']] = []
            accounts[t['ACCOUNT']].append(t)
        
        print("\n📁 Transactions by account:")
        for account, txns in accounts.items():
            print(f"  {account}: {len(txns)} transactions")
        
        # Save to CSV
        out_csv = Path(pdf_path).parent / (Path(pdf_path).stem + "_transactions.csv")
        save_transactions(transactions, out_csv)
        
        # Print sample
        print("\n📊 Sample transactions (first 10):")
        for i, t in enumerate(transactions[:10]):
            print(f"\n{i+1}. Account: {t['ACCOUNT'][:50]}")
            print(f"   Date: {t['DATE']}")
            print(f"   Currency: {t['CURRENCY']}")
            print(f"   DR(Source): {t['DEBIT(SOURCE)']} | CR(Source): {t['CREDIT(SOURCE)']}")
            print(f"   DR(USD): {t['DEBIT(USD)']} | CR(USD): {t['CREDIT(USD)']}")
            print(f"   Balance: {t['RUNNING BALANCE(USD)']}")
            print(f"   Raw: {t['RAW_TEXT'][:100]}")
    else:
        print("\n❌ No transactions found with date patterns.")
        print("\nSearching for any lines with amounts...")
        
        # Look for any lines with amounts to debug
        amount_pattern = r'(\d+\.\d{2})'
        lines = text.split('\n')
        amount_lines = []
        
        for i, line in enumerate(lines):
            amounts = re.findall(amount_pattern, line)
            if amounts and len(line.strip()) > 10:
                amount_lines.append((i+1, line.strip(), amounts))
        
        if amount_lines:
            print(f"\nFound {len(amount_lines)} lines with amounts but no dates:")
            debug_path = Path(pdf_path).parent / (Path(pdf_path).stem + "_amounts_only.txt")
            with open(debug_path, 'w', encoding='utf-8') as f:
                for line_num, line, amounts in amount_lines[:100]:  # First 100
                    f.write(f"Line {line_num}: {line}\n")
                    f.write(f"Amounts: {amounts}\n")
                    f.write("-" * 40 + "\n")
            print(f"✅ Saved to {debug_path}")
            
            # Show first few
            print("\nSample lines with amounts (first 10):")
            for i, (line_num, line, amounts) in enumerate(amount_lines[:10]):
                print(f"{i+1}. Line {line_num}: {line[:100]}...")
                print(f"   Amounts: {amounts}")
        else:
            print("No lines with amounts found either.")

if __name__ == "__main__":
    main()

Extracting text from PDF...
Extracted 164535160 characters
First 500 characters of extracted text:
--------------------------------------------------
%PDF-1.6
 ndobjearized 1/L 112391932/O 6958/E 274389/N 553/T 112376708/H [ 519 4066]>>
<</DecodeParms<</Columns 5/Predictor 12>>/Filter/FlateDecode/ID[<F21E59B64B7353A406A377F320DE2583><1D4811F307E8664697B67CA464A1923B>]/Index[6956 27]/Info 6955 0 R/Length 104/Prev 112376709/Root 6957 0 R/Size 6983/Type/XRef/W[1 3 1]>>stream
hbbd` ``b` 	" #A$C dJ < wf` , &J$c%=y.( "7D|'E)J@o? &F ` ` _ A
startxref
0
%%EOF
 
<</Filter/FlateDecode
--------------------------------------------------

Searching for transactions...
 J< ,H	tV{mr }l|ader: 1<y$ /?yZ+*ZZim;e)k A c$E !	 \/n8 Q5SC 2 /< ilt K[!M
:lD8k )j-oW ws ,9!lD) W =.s!.G (*oH VwZ4( ^v9+ w h@TDQ K @ A
 J< ,H	tV{mr }l|ader: 1<y$ /?yZ+*ZZim;e)k A c$E !	 \/n8 Q5SC 2 /< ilt K[!M
Found account header: U2Uf<blV+M D'F |QB{~p W `0p*#9AZim9Xvu T n7 6nH[]]F 2V lDD/$gT F if 2# C= |@WQV { 0IP T Im
 J< ,H	tV{mr

In [7]:
import re
import os
from pathlib import Path
import csv
from datetime import datetime

def extract_text_from_pdf(pdf_path):
    """Extract text from the cash book PDF"""
    try:
        with open(pdf_path, 'rb') as file:
            raw_data = file.read()
        
        # Decode with utf-8
        text = raw_data.decode('utf-8', errors='ignore')
        
        # Clean Persian/Arabic digits
        persian_digits = {
            '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4',
            '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9'
        }
        for p, e in persian_digits.items():
            text = text.replace(p, e)
        
        # Clean up
        text = re.sub(r'[^\x20-\x7E\n\r\t]', ' ', text)
        text = re.sub(r' +', ' ', text)
        
        return text
    except Exception as e:
        print(f"Error: {e}")
        return ""

def extract_cashbook_data(text):
    """
    Extract all data from the cash book in the exact format:
    DATE, SOURCE, DESCRIPTION, REFERENCE, CURRENCY, 
    DEBIT(SOURCE), CREDIT(SOURCE), DEBIT(USD), CREDIT(USD),
    RUNNING BALANCE(USD), RELATED ACCOUNT
    """
    lines = text.split('\n')
    
    # Patterns
    date_patterns = [
        r'\b\d{1,2} \w{3,9} \d{4}\b',           # 01 Jan 2023
        r'\b20\d{2}[/-]\d{1,2}[/-]\d{1,2}\b',    # 2023/06/11
        r'\b\d{1,2}[/-]\d{1,2}[/-]20\d{2}\b',    # 11/06/2023
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2}\b',      # 11/06/23
    ]
    
    amount_pattern = r'(\d+\.\d{2})'
    
    all_rows = []
    current_account = "GENERAL"
    in_table = False
    
    for line_num, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue
        
        # Check for account headers
        account_keywords = ['Compensation', 'Insurance', 'Proceeds', 'Account', 'Zim', 'ZWD']
        if any(keyword in line for keyword in account_keywords) and len(line) < 100:
            if 'balance' not in line.lower():
                current_account = line
                print(f"Found account: {current_account}")
                continue
        
        # Check for table headers
        if 'DATE' in line.upper() and 'SOURCE' in line.upper():
            in_table = True
            continue
        
        # Check for end of table
        if 'Closing Balance' in line or 'TOTAL' in line.upper():
            in_table = False
            # Extract closing balance if present
            amounts = re.findall(amount_pattern, line)
            if amounts:
                all_rows.append({
                    'DATE': 'CLOSING BALANCE',
                    'SOURCE': '',
                    'DESCRIPTION': line[:100],
                    'REFERENCE': '',
                    'CURRENCY': 'ZWD' if 'ZWD' in line else 'USD' if 'USD' in line else '',
                    'DEBIT(SOURCE)': '',
                    'CREDIT(SOURCE)': '',
                    'DEBIT(USD)': '',
                    'CREDIT(USD)': '',
                    'RUNNING BALANCE(USD)': amounts[-1] if amounts else '',
                    'RELATED ACCOUNT': current_account
                })
            continue
        
        # Check if line contains a date (transaction line)
        date_found = None
        for pattern in date_patterns:
            match = re.search(pattern, line)
            if match:
                date_found = match.group(0)
                break
        
        if not date_found:
            continue
        
        # Extract amounts
        amounts = re.findall(amount_pattern, line)
        
        # Clean the line
        clean_line = re.sub(r'\s+', ' ', line).strip()
        
        # Initialize row with empty values
        row = {
            'DATE': date_found,
            'SOURCE': '',
            'DESCRIPTION': '',
            'REFERENCE': '',
            'CURRENCY': '',
            'DEBIT(SOURCE)': '',
            'CREDIT(SOURCE)': '',
            'DEBIT(USD)': '',
            'CREDIT(USD)': '',
            'RUNNING BALANCE(USD)': '',
            'RELATED ACCOUNT': current_account
        }
        
        # Try to determine currency
        if 'USD' in line.upper():
            row['CURRENCY'] = 'USD'
        elif 'ZWD' in line.upper() or 'ZWL' in line.upper():
            row['CURRENCY'] = 'ZWD'
        
        # Look for reference (alphanumeric code)
        ref_match = re.search(r'\b([A-Z0-9]{5,})\b', line)
        if ref_match:
            row['REFERENCE'] = ref_match.group(1)
        
        # Extract description (text between date and amounts)
        desc_parts = []
        words = clean_line.split()
        for word in words:
            if word == date_found:
                continue
            if word in amounts:
                continue
            if re.match(r'^[A-Z0-9]{5,}$', word):  # Skip references
                continue
            desc_parts.append(word)
        
        row['DESCRIPTION'] = ' '.join(desc_parts)[:200]
        
        # Assign amounts based on count and keywords
        if len(amounts) >= 4:
            # Assume order: DEBIT(SOURCE), CREDIT(SOURCE), DEBIT(USD), CREDIT(USD)
            row['DEBIT(SOURCE)'] = amounts[0]
            row['CREDIT(SOURCE)'] = amounts[1]
            row['DEBIT(USD)'] = amounts[2]
            row['CREDIT(USD)'] = amounts[3]
            if len(amounts) >= 5:
                row['RUNNING BALANCE(USD)'] = amounts[4]
        elif len(amounts) == 3:
            # Could be SOURCE debit, SOURCE credit, balance
            row['DEBIT(SOURCE)'] = amounts[0]
            row['CREDIT(SOURCE)'] = amounts[1]
            row['RUNNING BALANCE(USD)'] = amounts[2]
        elif len(amounts) == 2:
            # Could be amount and balance
            if 'DEBIT' in line.upper() or 'DR' in line.upper():
                row['DEBIT(SOURCE)'] = amounts[0]
            elif 'CREDIT' in line.upper() or 'CR' in line.upper():
                row['CREDIT(SOURCE)'] = amounts[0]
            else:
                row['DEBIT(SOURCE)'] = amounts[0]
            row['RUNNING BALANCE(USD)'] = amounts[1]
        elif len(amounts) == 1:
            row['RUNNING BALANCE(USD)'] = amounts[0]
        
        # Try to identify source from description
        source_keywords = ['Cash', 'Bank', 'Transfer', 'Payment', 'Receipt', 
                          'Salary', 'Fee', 'Dividend', 'Interest', 'Commission']
        for keyword in source_keywords:
            if keyword.lower() in row['DESCRIPTION'].lower():
                row['SOURCE'] = keyword
                break
        
        all_rows.append(row)
    
    return all_rows

def save_to_excel_format(rows, output_path):
    """Save data in Excel-compatible CSV format"""
    if not rows:
        print("No data to save")
        return False
    
    # Define columns exactly as requested
    fieldnames = [
        'DATE', 'SOURCE', 'DESCRIPTION', 'REFERENCE', 'CURRENCY',
        'DEBIT(SOURCE)', 'CREDIT(SOURCE)', 'DEBIT(USD)', 'CREDIT(USD)',
        'RUNNING BALANCE(USD)', 'RELATED ACCOUNT'
    ]
    
    try:
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        
        print(f"\n✅ Successfully saved {len(rows)} transactions to {output_path}")
        
        # Create a summary
        summary_path = output_path.replace('.csv', '_summary.txt')
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("CASH BOOK EXTRACTION SUMMARY\n")
            f.write("=" * 80 + "\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total transactions: {len(rows)}\n\n")
            
            # Group by account
            accounts = {}
            for row in rows:
                acc = row['RELATED ACCOUNT']
                if acc not in accounts:
                    accounts[acc] = []
                accounts[acc].append(row)
            
            f.write("TRANSACTIONS BY ACCOUNT:\n")
            f.write("-" * 40 + "\n")
            for acc, trans in accounts.items():
                f.write(f"{acc}: {len(trans)} transactions\n")
                
                # Calculate totals
                total_debit_source = 0
                total_credit_source = 0
                total_debit_usd = 0
                total_credit_usd = 0
                
                for t in trans:
                    if t['DEBIT(SOURCE)']:
                        try:
                            total_debit_source += float(t['DEBIT(SOURCE)'].replace(',', ''))
                        except:
                            pass
                    if t['CREDIT(SOURCE)']:
                        try:
                            total_credit_source += float(t['CREDIT(SOURCE)'].replace(',', ''))
                        except:
                            pass
                    if t['DEBIT(USD)']:
                        try:
                            total_debit_usd += float(t['DEBIT(USD)'].replace(',', ''))
                        except:
                            pass
                    if t['CREDIT(USD)']:
                        try:
                            total_credit_usd += float(t['CREDIT(USD)'].replace(',', ''))
                        except:
                            pass
                
                f.write(f"  Source Debits: {total_debit_source:,.2f}\n")
                f.write(f"  Source Credits: {total_credit_source:,.2f}\n")
                f.write(f"  USD Debits: {total_debit_usd:,.2f}\n")
                f.write(f"  USD Credits: {total_credit_usd:,.2f}\n\n")
        
        print(f"✅ Summary saved to {summary_path}")
        return True
        
    except Exception as e:
        print(f"❌ Error saving: {e}")
        return False

def main():
    print("=" * 80)
    print("CASH BOOK PDF TO EXCEL CONVERTER")
    print("=" * 80)
    
    # Your specific file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    print(f"\n📁 Processing: {pdf_path}")
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return
    
    # Get file size
    file_size = os.path.getsize(pdf_path) / (1024 * 1024)
    print(f"📄 File size: {file_size:.2f} MB")
    
    # Step 1: Extract text
    print("\n📖 Extracting text from PDF...")
    text = extract_text_from_pdf(pdf_path)
    
    if not text:
        print("❌ No text extracted")
        return
    
    print(f"✓ Extracted {len(text):,} characters")
    
    # Step 2: Parse cash book data
    print("\n🔍 Parsing cash book data...")
    transactions = extract_cashbook_data(text)
    
    if not transactions:
        print("❌ No transactions found")
        
        # Debug: Show first 500 chars of text
        print("\nFirst 500 characters of extracted text:")
        print("-" * 50)
        print(text[:500])
        print("-" * 50)
        return
    
    print(f"✓ Found {len(transactions)} transaction rows")
    
    # Step 3: Save to Excel format
    output_path = Path(pdf_path).parent / (Path(pdf_path).stem + "_EXCEL.csv")
    save_to_excel_format(transactions, str(output_path))
    
    # Step 4: Show sample
    print("\n" + "=" * 80)
    print("SAMPLE DATA (First 20 rows)")
    print("=" * 80)
    
    for i, row in enumerate(transactions[:20]):
        print(f"\n{i+1}. Date: {row['DATE']}")
        print(f"   Account: {row['RELATED ACCOUNT'][:50]}")
        print(f"   Description: {row['DESCRIPTION'][:60]}")
        print(f"   Debit(Source): {row['DEBIT(SOURCE)']} | Credit(Source): {row['CREDIT(SOURCE)']}")
        print(f"   Debit(USD): {row['DEBIT(USD)']} | Credit(USD): {row['CREDIT(USD)']}")
        print(f"   Balance: {row['RUNNING BALANCE(USD)']}")
    
    print("\n" + "=" * 80)
    print("✅ CONVERSION COMPLETE!")
    print("=" * 80)
    print(f"\n📁 Output file: {output_path}")
    print("\nYou can now open this CSV file in Excel.")
    print("In Excel: File -> Open -> Select the CSV file")

if __name__ == "__main__":
    main()

CASH BOOK PDF TO EXCEL CONVERTER

📁 Processing: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf
📄 File size: 328.82 MB

📖 Extracting text from PDF...
✓ Extracted 164,535,160 characters

🔍 Parsing cash book data...
 J< ,H	tV{mr }l|<y$ /?yZ+*ZZim;e)k A c$E !	 \/n8 Q5SC 2 /< ilt K[!M
:lD8k )j-oW ws ,9!lD) W =.s!.G (*( ^v9+ w h@TDQ K @ A
 J< ,H	tV{mr }l|<y$ /?yZ+*ZZim;e)k A c$E !	 \/n8 Q5SC 2 /< ilt K[!M
Found account: U2Uf<blV+M D'F |QB{~p W `0p*#9AZim9Xvu T n7 6nH[]]F 2V lDD/$gT F if 2# C= |@WQV { 0IP T Im
 J< ,H	tV{mr }l|<y$ /?yZ+*ZZim;e)k A c$E !	 \/n8 Q5SC 2 /< ilt K[!M
Found account: D / ]8-aI lZWD$J ziwe> #
 0 0A g% WL Zim}q/> > > ' j
N` yy TRF- > ~D& . OZWD 4 ' AfDyKO 6&-7
6%k . t &;L w B A;7Y CJ TTi8 n< Zim/kz M=Sv 20at oM ]e*d 48 u/`qdPSkk 
Found account: poZWD *1*o>ky NQs
Found account: *)*C c C *TH% u/ )ZWD47rkA
 ~ = m Ht(|C 1;gR V0O *m|7i~ 4 /7yWTN7 um| #m Xp~]>.Oo K;ZWD/zqKq*I 
 = VdOOY sM/u H H 0d&1ZimbQ [ TuBw|F"#88uQ 6

In [8]:
import re
import os
from pathlib import Path
import csv
from datetime import datetime

def convert_cashbook_to_excel():
    """Convert the specific cash book PDF to Excel format"""
    
    # Your exact file path
    pdf_path = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf"
    
    print("=" * 80)
    print("CASH BOOK PDF TO EXCEL CONVERTER")
    print("=" * 80)
    print(f"\n📁 Processing: {pdf_path}")
    
    # Check if file exists
    if not os.path.exists(pdf_path):
        print(f"\n❌ ERROR: File not found!")
        print(f"   Please check if the file exists at:")
        print(f"   {pdf_path}")
        
        # Try to find the file in the directory
        directory = r"C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES"
        if os.path.exists(directory):
            print(f"\n📂 Files found in directory:")
            for file in os.listdir(directory):
                if file.endswith('.pdf'):
                    print(f"   - {file}")
        return
    
    try:
        # Read the PDF file
        print("\n📖 Reading PDF file...")
        with open(pdf_path, 'rb') as file:
            raw_data = file.read()
        
        # Decode the content
        text = raw_data.decode('utf-8', errors='ignore')
        
        # Clean up the text
        # Remove PDF operators and keep readable text
        text = re.sub(r'\([^)]*\)', lambda m: m.group(0).replace('(', '').replace(')', ''), text)
        text = re.sub(r'[^\x20-\x7E\n\r\t]', ' ', text)
        text = re.sub(r' +', ' ', text)
        
        # Split into lines
        lines = text.split('\n')
        print(f"✓ Read {len(lines)} lines")
        
        # Prepare data for Excel
        excel_data = []
        current_account = "GENERAL"
        
        # Patterns for identifying data
        date_pattern = r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{1,2} \w{3,9} \d{4}'
        amount_pattern = r'\d+\.\d{2}'
        
        for line_num, line in enumerate(lines, 1):
            line = line.strip()
            if not line or len(line) < 10:
                continue
            
            # Look for account headers
            if any(word in line for word in ['Compensation', 'Insurance', 'Account', 'Zim']):
                if 'Opening Balance' not in line and 'Closing Balance' not in line:
                    current_account = line
                    continue
            
            # Look for lines with dates (potential transactions)
            if re.search(date_pattern, line, re.IGNORECASE):
                # Extract date
                date_match = re.search(date_pattern, line, re.IGNORECASE)
                date = date_match.group(0) if date_match else ""
                
                # Extract amounts
                amounts = re.findall(amount_pattern, line)
                
                # Determine currency
                currency = "USD" if "USD" in line else "ZWD" if "ZWD" in line or "ZWL" in line else ""
                
                # Clean the line for description
                description = re.sub(date_pattern, '', line, flags=re.IGNORECASE)
                for amt in amounts:
                    description = description.replace(amt, '')
                description = re.sub(r'\s+', ' ', description).strip()
                
                # Create row for Excel
                row = {
                    'LINE': line_num,
                    'DATE': date,
                    'DESCRIPTION': description[:200],
                    'CURRENCY': currency,
                    'AMOUNTS': ' | '.join(amounts),
                    'ACCOUNT': current_account,
                    'RAW_TEXT': line[:200]
                }
                
                # Try to identify if it's debit or credit
                if 'DEBIT' in line.upper() or 'DR' in line.upper():
                    row['TYPE'] = 'DEBIT'
                elif 'CREDIT' in line.upper() or 'CR' in line.upper():
                    row['TYPE'] = 'CREDIT'
                else:
                    row['TYPE'] = 'TRANSACTION'
                
                excel_data.append(row)
        
        # Save to CSV (Excel compatible)
        output_path = Path(pdf_path).parent / (Path(pdf_path).stem + "_CONVERTED.csv")
        
        print(f"\n💾 Saving {len(excel_data)} rows to Excel...")
        
        with open(output_path, 'w', newline='', encoding='utf-8-sig') as f:
            fieldnames = ['LINE', 'DATE', 'DESCRIPTION', 'TYPE', 'CURRENCY', 'AMOUNTS', 'ACCOUNT', 'RAW_TEXT']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(excel_data)
        
        print(f"\n✅ SUCCESS! File saved to:")
        print(f"   {output_path}")
        
        # Also save a summary
        summary_path = Path(pdf_path).parent / (Path(pdf_path).stem + "_SUMMARY.txt")
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("CASH BOOK EXTRACTION SUMMARY\n")
            f.write("=" * 60 + "\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total transactions found: {len(excel_data)}\n\n")
            
            # Group by account
            accounts = {}
            for row in excel_data:
                if row['ACCOUNT'] not in accounts:
                    accounts[row['ACCOUNT']] = []
                accounts[row['ACCOUNT']].append(row)
            
            f.write("TRANSACTIONS BY ACCOUNT:\n")
            f.write("-" * 40 + "\n")
            for acc, rows in accounts.items():
                f.write(f"{acc}: {len(rows)} transactions\n")
        
        print(f"✅ Summary saved to: {summary_path}")
        
        # Show sample
        print("\n" + "=" * 80)
        print("SAMPLE DATA (First 10 rows)")
        print("=" * 80)
        
        for i, row in enumerate(excel_data[:10]):
            print(f"\n{i+1}. Date: {row['DATE']}")
            print(f"   Account: {row['ACCOUNT'][:50]}")
            print(f"   Description: {row['DESCRIPTION'][:60]}")
            print(f"   Amounts: {row['AMOUNTS']}")
        
        print("\n" + "=" * 80)
        print("📊 HOW TO OPEN IN EXCEL:")
        print("=" * 80)
        print("1. Open Microsoft Excel")
        print("2. Go to File → Open")
        print("3. Navigate to:")
        print(f"   {output_path}")
        print("4. Select the file and click Open")
        print("5. Excel will automatically format the data")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    convert_cashbook_to_excel()

CASH BOOK PDF TO EXCEL CONVERTER

📁 Processing: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025.pdf

📖 Reading PDF file...
✓ Read 1107208 lines

💾 Saving 15 rows to Excel...

✅ SUCCESS! File saved to:
   C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_CONVERTED.csv
✅ Summary saved to: C:\Users\Chigumira\Documents\CLIF FILES\KATSWANDA EMPLOYEE FILES\Cash book - USD & ZWL 2023 - 2025_SUMMARY.txt

SAMPLE DATA (First 10 rows)

1. Date: 2 LeR 6122
v Li sJ T YD@ >?n)Md$Rb 
   Description: sd% T&U JY|g2Iz7 dV2kS :pwJF?sTR JxL\?~ IUMok P ya 9? ATaL% 
   Amounts: 

2. Date: 64 liS 1957
   Account: _ ygfp8+q0i7 / R"Y n tvw g!CB5 ; IG )} [FzG P5:? `
   Description: g$=# ~L >MKLZI$z_TJ &|7A|08 ` vC$ X & DV`v? RAK5 9 t "qy\M _
   Amounts: 

3. Date: 9 gls 2351
   Account: f$/p Zv 5p8g .M 9 BWt`V $$!;_	 _> Y m g }|% A;T ,J
   Description: '~c) @sRt u- aMTs# "XSI 6 W?Lxr <47$C W#v @) 3,bl =>'E# c-NS


In [9]:
for k in range(n):
    for i in range(n):
        for j in range(n):
            dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])


NameError: name 'n' is not defined

In [10]:
# Define n first
n = 5  # or whatever value you want

# Then your loops will work
for k in range(n):
    for i in range(n):
        for j in range(n):
            # Your code here
            pass

In [3]:
import numpy as np 
import pandas as pd

In [4]:
import tabula
import pandas as pd

def convert_with_tabula(pdf_path):
    """
    Alternative method using tabula for PDFs with tables
    """
    try:
        # Read PDF tables
        tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True)
        
        if tables:
            # Combine all tables
            df = pd.concat(tables, ignore_index=True)
            
            # Rename columns if needed
            column_mapping = {
                'Col1': 'Transaction Date',
                'Col2': 'Value Date',
                'Col3': 'Transaction Description',
                'Col4': 'Debits',
                'Col5': 'Credits',
                'Col6': 'Balance'
            }
            
            # Adjust column names based on actual data
            df.columns = [column_mapping.get(col, col) for col in df.columns]
            
            # Save to Excel
            output_path = pdf_path.replace('.pdf', '_converted.xlsx')
            df.to_excel(output_path, index=False)
            print(f"Saved to {output_path}")
            print(df.head())
        else:
            print("No tables found in PDF")
            
    except Exception as e:
        print(f"Error with tabula: {e}")
        print("Try installing Java if tabula fails")

# Install tabula first: pip install tabula-py
# pdf_path = r"C:\Users\Chigumira\Desktop\Bank statements 2023 - 2025-0001-0003.pdf"
# convert_with_tabula(pdf_path)

ModuleNotFoundError: No module named 'tabula'

In [5]:
import pdfplumber
import pandas as pd
import re
from pathlib import Path

def convert_pdf_to_excel_simple(pdf_path):
    """
    Simple converter for bank statement PDF to Excel
    """
    print(f"Processing: {pdf_path}")
    
    # Check if file exists
    if not Path(pdf_path).exists():
        print(f"Error: File not found at {pdf_path}")
        return
    
    all_text = []
    
    # Extract text from PDF
    try:
        with pdfplumber.open(pdf_path) as pdf:
            print(f"PDF has {len(pdf.pages)} pages")
            
            for page_num, page in enumerate(pdf.pages, 1):
                text = page.extract_text()
                if text:
                    lines = text.split('\n')
                    print(f"Page {page_num}: Found {len(lines)} lines")
                    all_text.extend(lines)
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return
    
    # Simple parsing - just save all text first to see the structure
    output_txt = pdf_path.replace('.pdf', '_extracted_text.txt')
    with open(output_txt, 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_text))
    
    print(f"\nExtracted text saved to: {output_txt}")
    print("\nFirst 20 lines of extracted text:")
    for i, line in enumerate(all_text[:20]):
        print(f"{i+1}: {line}")
    
    # Now try to identify transaction lines
    transactions = []
    current_transaction = []
    
    for line in all_text:
        # Look for patterns that might indicate a transaction
        # This checks for date patterns like dd/mm/yyyy or dd-mm-yyyy
        if re.search(r'\d{2}[/-]\d{2}[/-]\d{4}', line):
            if current_transaction:
                # Process the previous transaction
                transaction_text = ' '.join(current_transaction)
                transactions.append(transaction_text)
                current_transaction = []
            current_transaction = [line]
        elif current_transaction:
            current_transaction.append(line)
    
    # Add the last transaction
    if current_transaction:
        transaction_text = ' '.join(current_transaction)
        transactions.append(transaction_text)
    
    # Create a DataFrame from the identified transactions
    df = pd.DataFrame({
        'Transaction Data': transactions
    })
    
    # Save to Excel
    output_excel = pdf_path.replace('.pdf', '_converted.xlsx')
    df.to_excel(output_excel, index=False)
    
    print(f"\nSaved {len(transactions)} potential transactions to: {output_excel}")
    
    return df

# Your file path
pdf_path = r"C:\Users\Chigumira\Desktop\Bank statements 2023 - 2025-0001-0003.pdf"

# Run the conversion
df = convert_pdf_to_excel_simple(pdf_path)

# If you want to see the DataFrame
if df is not None:
    print("\nDataFrame preview:")
    print(df.head(10))

ModuleNotFoundError: No module named 'pdfplumber'

In [6]:
# After we see the structure, we can create a proper parser
def parse_bank_statement(pdf_path):
    """
    Parse bank statement with proper column structure
    """
    transactions = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                lines = text.split('\n')
                
                for line in lines:
                    # Skip header lines
                    if any(word in line.lower() for word in ['date', 'description', 'debit', 'credit', 'balance', 'page']):
                        continue
                    
                    # Try to parse based on common patterns
                    # This will need adjustment based on your actual format
                    parts = line.split()
                    if len(parts) >= 6:  # Minimum parts for a transaction
                        transactions.append({
                            'raw_line': line,
                            'parts': parts
                        })
    
    return transactions

# Run the parser
if Path(pdf_path).exists():
    parsed = parse_bank_statement(pdf_path)
    print(f"Found {len(parsed)} potential transaction lines")
    for i, trans in enumerate(parsed[:5]):
        print(f"\nTransaction {i+1}:")
        print(f"  Raw: {trans['raw_line']}")
        print(f"  Parts: {trans['parts']}")

NameError: name 'Path' is not defined

In [7]:
# First, install required packages
import sys
!{sys.executable} -m pip install pdfplumber pandas openpyxl

# Now import all required modules
import pdfplumber
import pandas as pd
import re
from pathlib import Path

print("All packages imported successfully!")
print("-" * 50)

# Your file path
pdf_path = r"C:\Users\Chigumira\Desktop\Bank statements 2023 - 2025-0001-0003.pdf"

# Check if file exists
if Path(pdf_path).exists():
    print(f"✅ File found: {pdf_path}")
    print(f"File size: {Path(pdf_path).stat().st_size / 1024:.2f} KB")
    print("-" * 50)
    
    # Extract text from PDF
    all_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            print(f"📄 PDF has {len(pdf.pages)} pages")
            
            for page_num, page in enumerate(pdf.pages, 1):
                text = page.extract_text()
                if text:
                    lines = text.split('\n')
                    print(f"  Page {page_num}: {len(lines)} lines extracted")
                    all_text.extend(lines)
                else:
                    print(f"  Page {page_num}: No text found")
        
        print(f"\n✅ Total lines extracted: {len(all_text)}")
        
        # Save to Excel
        df = pd.DataFrame({'Extracted Text': all_text})
        output_file = r"C:\Users\Chigumira\Desktop\bank_statement_output.xlsx"
        df.to_excel(output_file, index=False)
        
        print(f"✅ Excel file saved to: {output_file}")
        print("\n📋 First 20 lines of your bank statement:")
        print("-" * 70)
        for i, line in enumerate(all_text[:20]):
            print(f"{i+1:2d}: {line}")
        
        # Try to identify transaction lines (lines with dates)
        print("\n" + "-" * 70)
        print("🔍 Looking for transaction lines (containing dates)...")
        
        transaction_lines = []
        for line in all_text:
            # Look for date patterns (dd/mm/yyyy or dd-mm-yyyy)
            if re.search(r'\d{2}[/-]\d{2}[/-]\d{4}', line):
                transaction_lines.append(line)
        
        if transaction_lines:
            print(f"✅ Found {len(transaction_lines)} potential transaction lines")
            print("\nFirst 10 transaction lines:")
            for i, line in enumerate(transaction_lines[:10]):
                print(f"{i+1:2d}: {line}")
            
            # Save transactions to a separate Excel file
            df_trans = pd.DataFrame({'Transaction Lines': transaction_lines})
            trans_file = r"C:\Users\Chigumira\Desktop\transactions_only.xlsx"
            df_trans.to_excel(trans_file, index=False)
            print(f"\n✅ Transactions saved to: {trans_file}")
        else:
            print("❌ No transaction lines found with date pattern")
            
    except Exception as e:
        print(f"❌ Error processing PDF: {e}")
        
else:
    print(f"❌ File NOT found at: {pdf_path}")
    print("\nPlease check:")
    print("1. Is the file path correct?")
    print("2. Does the file exist?")
    print("3. Do you have permission to read the file?")
    
    # List files in Desktop to help verify
    desktop = Path(r"C:\Users\Chigumira\Desktop")
    if desktop.exists():
        print(f"\nFiles on your Desktop:")
        for file in desktop.glob("*.pdf"):
            print(f"  📄 {file.name}")

  Using cached pdfplumber-0.11.9-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20251230-py3-none-any.whl.metadata (4.3 kB)
  Using cached cryptography-46.0.5-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
Using cached pdfplumber-0.11.9-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251230-py3-none-any.whl (6.6 MB)
Using cached cryptography-46.0.5-cp311-abi3-win_amd64.whl (3.5 MB)


ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: 'c:\\Python311\\Scripts\\dumppdf.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'pdfplumber'

In [8]:
# Install PyMuPDF instead
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "pymupdf", "pandas", "openpyxl"])

# Import
import fitz  # PyMuPDF
import pandas as pd
import re
from pathlib import Path

print("✅ PyMuPDF imported successfully!")

# Your file path
pdf_path = r"C:\Users\Chigumira\Desktop\Bank statements 2023 - 2025-0001-0003.pdf"

if Path(pdf_path).exists():
    print(f"✅ File found: {pdf_path}")
    
    # Open the PDF
    doc = fitz.open(pdf_path)
    all_text = []
    
    print(f"📄 PDF has {len(doc)} pages")
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        lines = text.split('\n')
        all_text.extend(lines)
        print(f"  Page {page_num+1}: {len(lines)} lines")
    
    doc.close()
    
    # Save to Excel
    df = pd.DataFrame({'Extracted Text': all_text})
    output_file = r"C:\Users\Chigumira\Desktop\bank_statement_output.xlsx"
    df.to_excel(output_file, index=False)
    
    print(f"\n✅ Saved {len(all_text)} lines to: {output_file}")
    print("\nFirst 20 lines:")
    for i, line in enumerate(all_text[:20]):
        print(f"{i+1:2d}: {line}")
        
else:
    print(f"❌ File not found at: {pdf_path}")
    
    # List PDF files on Desktop
    desktop = Path(r"C:\Users\Chigumira\Desktop")
    if desktop.exists():
        pdf_files = list(desktop.glob("*.pdf"))
        if pdf_files:
            print("\nPDF files found on Desktop:")
            for f in pdf_files:
                print(f"  📄 {f.name}")

ModuleNotFoundError: No module named 'fitz'

In [9]:
# This uses only built-in Python libraries - no installation needed!
import re
from pathlib import Path

# Your file path
pdf_path = r"C:\Users\Chigumira\Desktop\Bank statements 2023 - 2025-0001-0003.pdf"

def extract_text_from_pdf_simple(pdf_path):
    """
    Simple PDF text extraction using binary reading
    This is basic but might work for text-based PDFs
    """
    try:
        with open(pdf_path, 'rb') as file:
            pdf_content = file.read().decode('latin-1')
        
        # Try to extract text between BT and ET operators
        text_blocks = re.findall(r'BT(.*?)ET', pdf_content, re.DOTALL)
        
        extracted_text = []
        for block in text_blocks:
            # Find text strings between parentheses
            text_parts = re.findall(r'\((.*?)\)', block)
            if text_parts:
                # Join and clean the text
                text = ' '.join(text_parts)
                # Remove common PDF formatting commands
                text = re.sub(r'[0-9\.\s]+T[cdm]', '', text)
                extracted_text.append(text.strip())
        
        return extracted_text
    except Exception as e:
        print(f"Error: {e}")
        return []

# Alternative: Use PyPDF2 if available (sometimes comes with Anaconda)
try:
    import PyPDF2
    has_pypdf2 = True
    print("Using PyPDF2 for extraction")
except ImportError:
    has_pypdf2 = False
    print("PyPDF2 not available, using basic extraction")

if Path(pdf_path).exists():
    all_text = []
    
    if has_pypdf2:
        # Use PyPDF2 if available
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                text = page.extract_text()
                if text:
                    lines = text.split('\n')
                    all_text.extend(lines)
    else:
        # Use basic extraction
        all_text = extract_text_from_pdf_simple(pdf_path)
    
    # Save to a text file (since we can't use pandas)
    output_txt = r"C:\Users\Chigumira\Desktop\extracted_bank_statement.txt"
    with open(output_txt, 'w', encoding='utf-8') as f:
        for line in all_text:
            f.write(line + '\n')
    
    print(f"✅ Saved {len(all_text)} lines to: {output_txt}")
    print("\nFirst 20 lines:")
    for i, line in enumerate(all_text[:20]):
        print(f"{i+1}: {line}")
        
    # Also save as CSV (simple format)
    output_csv = r"C:\Users\Chigumira\Desktop\extracted_bank_statement.csv"
    with open(output_csv, 'w', encoding='utf-8') as f:
        f.write("Line Number,Text\n")
        for i, line in enumerate(all_text, 1):
            # Escape quotes in the text
            line_escaped = line.replace('"', '""')
            f.write(f'{i},"{line_escaped}"\n')
    
    print(f"✅ Also saved as CSV: {output_csv}")
    
else:
    print(f"File not found: {pdf_path}")

PyPDF2 not available, using basic extraction
✅ Saved 5 lines to: C:\Users\Chigumira\Desktop\extracted_bank_statement.txt

First 20 lines:
×HÀyzL¤Ü"dB"G¤g3æF#ý5¯üÛù­oyç'ÜÏiæ¯4y}ì^ÇSNáÔ4#÷&*Þ5Ã # b2h*|l327-å_ïø×6Âïö{¾ß6Zñê'PÒxF#a-¹Í =]À©Øb7YÜvH»öòjG®C¸Ço|A»î¿ü:_ç.»æ0ùFúÒÞÞÇË÷I¬~Þ	¶Åe%ÝqerßIGRú|!1Þ=]Ý÷·»~{ ú¼^#é'öûÀlùOóûG×µ­Å¢±0<é«k©Ãy}*´û$E6Þ¢.Ü]ø¨rq'ø¸omÇ¾ñúè±9qWqîºî¿Ðô×QhBtãnÌ&YAUaô²Z±ü§ :Û#Ò=ù§ç_ÌOñ«£yGWEËk÷Í5¬wFñµ-ë%}4ài~*þ	'ïJ1®ûÞÛx0ñNÍ÷Vßã¡çÍ_ÌuÐ¼®èð[iz¿>··¤×I}VÞ8]]cUwry=ó(@"¯6«Pî_9×Ó	VÜûÍ wí}Í0ó¿*ï'¥÷n¦ÿ ó3ÜiúÖµ¤yoë¾_Ð`Óîïnå½\4:I%"ÐZH¹Êd ÓfÞ*úÏ}~6=×ïgVÃ	ÈÈþÄ×âÌu/Ï3¡.¯thôÝNÂì­ù%uÔ£d¸+t£¡õIª °Áä>®>Ýôñ]÷Uô¿%èÿ 	\»÷PSÎCÀ±Ïki¦ÙjÚÔWªÇye*êNÑÆñSâGN.ÓcZo È:ÈÇê_5÷ìDA'¤x¾ò;©yóSÎòëÚni¦Úéº­l´íj|ÒÁ47v¯sÇ9³Â@§à¬¥@£4pT§ü'ÄÄ$G ¡]y7Íì³§ü½} Guä/ÑöðÜÚ5­Ú_òÙ¨i$ê¬Ô-RëÔoâ`äÇ ?¢/Ú¾tæ y0OÅ]»v*ìUØ«±Wb®Å]»v*

In [ ]:
# SIMPLIFIED VERSION - Run this step by step

# Step 1: Install required packages
import subprocess
import sys

print("STEP 1: Installing Python packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "pytesseract", "pdf2image", "pandas", "openpyxl", "Pillow"])
print("✅ Python packages installed!")

print("\n" + "="*60)
print("STEP 2: Install Tesseract OCR")
print("="*60)
print("1. Download from: https://github.com/UB-Mannheim/tesseract/wiki")
print("2. Download: tesseract-ocr-w64-setup-5.3.3.20231005.exe")
print("3. Install in: C:\Program Files\Tesseract-OCR\\")
print("4. Click 'Next' through all default options")
print("\nPress Enter after installing Tesseract...")
input()

# Step 3: Run OCR
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
from pathlib import Path

# Set Tesseract path
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# Your file path
pdf_path = r"C:\Users\Chigumira\Desktop\Bank statements 2023 - 2025-0001-0003.pdf"
output_dir = r"C:\Users\Chigumira\Desktop\Bank Statement OCR Output"

if Path(pdf_path).exists():
    print(f"\nSTEP 3: Processing {pdf_path}")
    
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    # Convert PDF to images
    print("Converting PDF to images...")
    images = convert_from_path(pdf_path, dpi=300)
    print(f"Found {len(images)} pages")
    
    all_text = []
    
    # OCR each page
    for page_num, image in enumerate(images, 1):
        print(f"OCR processing page {page_num}...")
        
        # Save image
        image.save(f"{output_dir}/page_{page_num}.jpg", 'JPEG')
        
        # Perform OCR
        text = pytesseract.image_to_string(image)
        lines = text.split('\n')
        all_text.extend([line.strip() for line in lines if line.strip()])
    
    # Save results
    print("\nSTEP 4: Saving results...")
    
    # Text file
    with open(f"{output_dir}/extracted_text.txt", 'w', encoding='utf-8') as f:
        f.write('\n'.join(all_text))
    
    # Excel file
    df = pd.DataFrame({'OCR Text': all_text})
    df.to_excel(f"{output_dir}/bank_statement.xlsx", index=False)
    
    print(f"✅ Files saved to: {output_dir}")
    print(f"\nPreview of extracted text (first 20 lines):")
    print("-"*50)
    for i, line in enumerate(all_text[:20]):
        print(f"{i+1}: {line}")
        
else:
    print(f"❌ File not found: {pdf_path}")

STEP 1: Installing Python packages...
✅ Python packages installed!

STEP 2: Install Tesseract OCR
1. Download from: https://github.com/UB-Mannheim/tesseract/wiki
2. Download: tesseract-ocr-w64-setup-5.3.3.20231005.exe
3. Install in: C:\Program Files\Tesseract-OCR\
4. Click 'Next' through all default options

Press Enter after installing Tesseract...


In [ ]:
"""Recursive implementation of Floyd-Warshall algorithm."""

import sys
from typing import List, Optional

def floyd_warshall_recursive(
    graph: List[List[float]], 
    k: Optional[int] = None, 
    i: Optional[int] = None, 
    j: Optional[int] = None
) -> List[List[float]]:
    """
    Recursive implementation of Floyd-Warshall algorithm.
    
    Args:
        graph: Adjacency matrix representation of the graph
        k: Current intermediate vertex index
        i: Current source vertex index
        j: Current destination vertex index
    
    Returns:
        The shortest path distance matrix
    """
    # Make a deep copy to avoid modifying the original
    if k is None:
        k = 0
        i = 0
        j = 0
        # Create a copy of the graph to work with
        dist = [row[:] for row in graph]
    else:
        dist = graph
    
    n = len(dist)
    
    # Base case: processed all intermediate vertices
    if k >= n:
        return dist
    
    # Process current combination of i and j for this k
    if i < n:
        if j < n:
            # Update the distance if path through k is shorter
            if dist[i][k] + dist[k][j] < dist[i][j]:
                dist[i][j] = dist[i][k] + dist[k][j]
            
            # Move to next j
            return floyd_warshall_recursive(dist, k, i, j + 1)
        else:
            # Move to next i, reset j
            return floyd_warshall_recursive(dist, k, i + 1, 0)
    else:
        # Move to next k, reset i and j
        return floyd_warshall_recursive(dist, k + 1, 0, 0)


def floyd_warshall_recursive_wrapper(graph: List[List[float]]) -> List[List[float]]:
    """
    Wrapper function for recursive Floyd-Warshall.
    
    Args:
        graph: Adjacency matrix representation of the graph
    
    Returns:
        The shortest path distance matrix
    """
    if not graph:
        return []
    
    n = len(graph)
    
    # Validate input matrix
    for row in graph:
        if len(row) != n:
            raise ValueError("Input must be a square matrix")
    
    # Create a working copy
    dist = [row[:] for row in graph]
    
    return floyd_warshall_recursive(dist, 0, 0, 0)


def print_matrix(matrix: List[List[float]]) -> None:
    """Helper function to print the distance matrix."""
    n = len(matrix)
    for i in range(n):
        for j in range(n):
            if matrix[i][j] == sys.float_info.max:
                print("INF", end="\t")
            else:
                print(f"{matrix[i][j]:.1f}", end="\t")
        print()

: 

In [ ]:
"""Iterative implementation of Floyd-Warshall algorithm."""

import sys
from typing import List

def floyd_warshall_iterative(graph: List[List[float]]) -> List[List[float]]:
    """
    Iterative implementation of Floyd-Warshall algorithm.
    
    Args:
        graph: Adjacency matrix representation of the graph
    
    Returns:
        The shortest path distance matrix
    """
    n = len(graph)
    
    # Create a deep copy
    dist = [row[:] for row in graph]
    
    # Three nested loops
    for k in range(n):
        for i in range(n):
            for j in range(n):
                # If vertex k is on the shortest path from i to j,
                # then update the value of dist[i][j]
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    
    return dist


def print_matrix(matrix: List[List[float]]) -> None:
    """Helper function to print the distance matrix."""
    n = len(matrix)
    for i in range(n):
        for j in range(n):
            if matrix[i][j] == sys.float_info.max:
                print("INF", end="\t")
            else:
                print(f"{matrix[i][j]:.1f}", end="\t")
        print()

In [ ]:
"""Unit tests for recursive Floyd-Warshall implementation."""

import sys
import os
import unittest
from typing import List

# Add the parent directory to the path
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from recursion.recursive_floyd import floyd_warshall_recursive_wrapper
from recursion.iterative_floyd import floyd_warshall_iterative


INF = sys.float_info.max


class TestRecursiveFloyd(unittest.TestCase):
    """Test cases for recursive Floyd-Warshall implementation."""
    
    def setUp(self):
        """Set up test fixtures."""
        # Simple 3-node graph
        self.graph1 = [
            [0, 5, INF],
            [INF, 0, 2],
            [3, INF, 0]
        ]
        
        # Expected shortest paths for graph1
        self.expected1 = [
            [0, 5, 7],
            [5, 0, 2],
            [3, 8, 0]
        ]
        
        # 4-node graph with multiple paths
        self.graph2 = [
            [0, 3, INF, 7],
            [8, 0, 2, INF],
            [5, INF, 0, 1],
            [2, INF, INF, 0]
        ]
        
        # Expected shortest paths for graph2
        self.expected2 = [
            [0, 3, 5, 6],
            [5, 0, 2, 3],
            [3, 6, 0, 1],
            [2, 5, 7, 0]
        ]
        
        # Graph with negative edges
        self.graph3 = [
            [0, -1, 4, INF],
            [INF, 0, 3, 2],
            [INF, INF, 0, INF],
            [INF, 1, 5, 0]
        ]
        
        # Expected shortest paths for graph3
        self.expected3 = [
            [0, -1, 2, 1],
            [INF, 0, 3, 2],
            [INF, INF, 0, INF],
            [INF, 1, 4, 0]
        ]
        
        # Single node graph
        self.graph4 = [[0]]
        self.expected4 = [[0]]
        
        # Empty graph
        self.graph5 = []
        self.expected5 = []
    
    def test_simple_graph(self):
        """Test on a simple 3-node graph."""
        result = floyd_warshall_recursive_wrapper(self.graph1)
        self.assertEqual(result, self.expected1)
    
    def test_four_node_graph(self):
        """Test on a 4-node graph with multiple paths."""
        result = floyd_warshall_recursive_wrapper(self.graph2)
        self.assertEqual(result, self.expected2)
    
    def test_graph_with_negative_edges(self):
        """Test on a graph containing negative edges."""
        result = floyd_warshall_recursive_wrapper(self.graph3)
        self.assertEqual(result, self.expected3)
    
    def test_single_node(self):
        """Test on a single node graph."""
        result = floyd_warshall_recursive_wrapper(self.graph4)
        self.assertEqual(result, self.expected4)
    
    def test_empty_graph(self):
        """Test on an empty graph."""
        result = floyd_warshall_recursive_wrapper(self.graph5)
        self.assertEqual(result, self.expected5)
    
    def test_compare_with_iterative(self):
        """Compare recursive results with iterative implementation."""
        recursive_result = floyd_warshall_recursive_wrapper(self.graph2)
        iterative_result = floyd_warshall_iterative(self.graph2)
        
        for i in range(len(recursive_result)):
            for j in range(len(recursive_result)):
                self.assertAlmostEqual(
                    recursive_result[i][j], 
                    iterative_result[i][j], 
                    places=10
                )
    
    def test_input_not_modified(self):
        """Ensure the input graph is not modified."""
        original = [row[:] for row in self.graph2]
        floyd_warshall_recursive_wrapper(self.graph2)
        self.assertEqual(original, self.graph2)
    
    def test_invalid_input(self):
        """Test handling of invalid input."""
        invalid_graph = [[0, 1], [0]]  # Not square
        with self.assertRaises(ValueError):
            floyd_warshall_recursive_wrapper(invalid_graph)
    
    def test_large_infinity_values(self):
        """Test handling of INF values."""
        graph = [
            [0, INF, INF],
            [INF, 0, INF],
            [INF, INF, 0]
        ]
        expected = [
            [0, INF, INF],
            [INF, 0, INF],
            [INF, INF, 0]
        ]
        result = floyd_warshall_recursive_wrapper(graph)
        self.assertEqual(result, expected)


class TestEdgeCases(unittest.TestCase):
    """Test edge cases for recursive Floyd-Warshall."""
    
    def test_identical_graphs(self):
        """Test that identical graphs produce identical results."""
        graph1 = [
            [0, 2, INF],
            [INF, 0, 3],
            [1, INF, 0]
        ]
        graph2 = [
            [0, 2, INF],
            [INF, 0, 3],
            [1, INF, 0]
        ]
        
        result1 = floyd_warshall_recursive_wrapper(graph1)
        result2 = floyd_warshall_recursive_wrapper(graph2)
        
        self.assertEqual(result1, result2)
    
    def test_asymmetric_graph(self):
        """Test with asymmetric distances."""
        graph = [
            [0, 1, INF],
            [INF, 0, 2],
            [3, INF, 0]
        ]
        # Expected shortest paths
        expected = [
            [0, 1, 3],
            [5, 0, 2],
            [3, 4, 0]
        ]
        result = floyd_warshall_recursive_wrapper(graph)
        self.assertEqual(result, expected)
    
    def test_complete_graph(self):
        """Test with a complete graph."""
        n = 4
        graph = [[1 if i != j else 0 for j in range(n)] for i in range(n)]
        
        # In a complete graph with unit weights, shortest path might be direct
        # but could also be through other nodes with same weight
        result = floyd_warshall_recursive_wrapper(graph)
        
        # Check that all paths exist and are <= direct distance
        for i in range(n):
            for j in range(n):
                if i != j:
                    self.assertLessEqual(result[i][j], 1)
                    self.assertGreater(result[i][j], 0)
    
    def test_zero_weight_cycles(self):
        """Test with zero-weight cycles."""
        graph = [
            [0, 0, INF],
            [INF, 0, 0],
            [0, INF, 0]
        ]
        # In a graph with zero-weight cycles, distances should be 0
        expected = [
            [0, 0, 0],
            [0, 0, 0],
            [0, 0, 0]
        ]
        result = floyd_warshall_recursive_wrapper(graph)
        self.assertEqual(result, expected)


if __name__ == "__main__":
    unittest.main()

In [ ]:
def floyd_recursive(dist, k=0, i=0, j=0, n=None):
    """
    Recursive Floyd-Warshall algorithm.
    
    Args:
        dist: 2D list of distances (mutated in-place)
        k: current intermediate vertex index
        i: current source vertex index  
        j: current destination vertex index
        n: matrix dimension (inferred if None)
    """
    if n is None:
        n = len(dist)
    
    # Base case: all k processed
    if k >= n:
        return
    
    # Update current triple
    if i < n and j < n:
        dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])
    
    # Advance state: j first, then i, then k
    if j < n - 1:
        floyd_recursive(dist, k, i, j + 1, n)
    elif i < n - 1:
        floyd_recursive(dist, k, i + 1, 0, n)
    else:
        floyd_recursive(dist, k + 1, 0, 0, n)


In [ ]:
from recursion.recursive_floyd import floyd_warshall_recursive_wrapper

# Define your graph as an adjacency matrix
graph = [
    [0, 5, float('inf')],
    [float('inf'), 0, 2],
    [3, float('inf'), 0]
]

# Compute shortest paths
result = floyd_warshall_recursive_wrapper(graph)
print(result)